# Agentic RAG for Educator MCQs — Multiagent MVP

## Problem
Educators need high-quality, evidence-based multiple-choice questions (MCQs) for medical education, but current solutions lack:
- **Comprehensive research**: Limited to single sources (PubMed only)
- **Quality control**: No systematic validation or evaluation
- **Scalability**: Manual processes don't scale for curriculum development
- **Evidence grounding**: Questions often lack proper citations and rationale

## Solution
A **multiagentic hierarchical RAG system** that:
- **Supervisor Agent**: Orchestrates the entire workflow using LangGraph
- **Researcher Agents**: Parallel PubMed and Tavily agents for comprehensive data collection
- **Intelligent Processing**: Re-ranking, chunking, vector storage (Qdrant), and knowledge graph (SQLite)
- **MCQ Generation**: LLM-powered creation with strict constraints and validation
- **Quality Assurance**: RAGAS evaluation + rubric-based assessment

## Target Audience
- **Medical Educators**: Creating curriculum assessments
- **Educational Institutions**: Scaling MCQ development
- **Content Developers**: Building evidence-based educational materials
- **Researchers**: Studying educational assessment quality

## Multiagent Architecture Overview
```
START → Clarify → PubMedAgent → TavilyAgent → Merge + Re-rank → MCQ Generation → END
                ↗              ↘
            Supervisor ← → Researcher Agents
                ↓
            Final Report
```

## Notebook Outline

**0) Config** - Environment Setup  
**1) State Models** - Typed Containers for Docs and MCQs  
**2) PubMedAgent** - PubMed E-utilities Integration  
**3) TavilyAgent** - Web Search Integration  
**4) Merge + Re-rank** - Combine and Rank Results  
**5) MCQ Generation** - Create Questions from Ranked Results  
**6) End-to-End Run** - Complete Pipeline Execution  
**7) Chunk and Store** - Vector Database Integration  
**8) Mini Knowledge Graph** - Regex-based Concept Extraction  
**9) Retrieval Integration** - Semantic + Knowledge Graph  
**10) Hierarchical Orchestration** with LangGraph  
**11) MCQ Rubric & Checks** (Agent Node)  
**12) RAGAS Evaluation** (Qdrant-Grounded)  
**13) Advanced Retrieval** (Hybrid: Qdrant Dense + SQLite FTS5 Sparse with RRF)  
**14) Orchestration** with Swappable Retriever (LangGraph)  
**15) RAGAS Evaluation** (Retriever Comparison Mode)  
**16) Comparing retriver** ( Dense vs Hybrid (Dense + Sparse)


In [24]:
# Section 0: Config - Environment Setup
import os
from dotenv import load_dotenv

# Clear any cached environment variables
for key in list(os.environ.keys()):
    if key.startswith(('LANGCHAIN', 'LANGSMITH', 'OPENAI', 'TAVILY', 'COHERE')):
        del os.environ[key]

def load_config():
    """Load configuration from .env file and return settings with validation."""
    
    # Load environment variables
    load_dotenv()
    
    # Required API keys
    api_keys = {
        'OPENAI_API_KEY': os.getenv('OPENAI_API_KEY'),
        'TAVILY_API_KEY': os.getenv('TAVILY_API_KEY'), 
        'COHERE_API_KEY': os.getenv('COHERE_API_KEY'),
        'LANGCHAIN_API_KEY': os.getenv('LANGCHAIN_API_KEY')  # Match your .env file
    }
    
    # Boolean flags
    use_tavily = os.getenv('USE_TAVILY', 'True').lower() == 'true'
    use_stubs = os.getenv('USE_STUBS', 'False').lower() == 'true'
    
    # Check for missing keys and determine stub usage
    missing_keys = [key for key, value in api_keys.items() if not value]
    
    # If any key is missing, use stubs for those services
    if missing_keys:
        print(f"Missing API keys: {', '.join(missing_keys)}")
        print("Will use stub mode for missing services")
        use_stubs = True
    
    return api_keys, use_tavily, use_stubs, missing_keys

# Load and display configuration
try:
    api_keys, use_tavily, use_stubs, missing_keys = load_config()
    
    print("Configuration Loaded Successfully!")
    print(f"Tavily Enabled: {use_tavily}")
    print(f"Stub Mode: {use_stubs}")
    print()
    
    # Display settings table (without revealing any API key digits)
    print("Settings Summary:")
    print("-" * 50)
    print(f"{'Setting':<20} {'Value':<15} {'Status':<10}")
    print("-" * 50)
    
    for key in ['OPENAI_API_KEY', 'TAVILY_API_KEY', 'COHERE_API_KEY', 'LANGCHAIN_API_KEY']:
        value = '****' if api_keys[key] else 'MISSING'
        status = '✓' if api_keys[key] else '✗'
        print(f"{key:<20} {value:<15} {status:<10}")
    
    print(f"{'USE_TAVILY':<20} {use_tavily:<15} {'✓' if use_tavily else '✗':<10}")
    print(f"{'USE_STUBS':<20} {use_stubs:<15} {'✓' if use_stubs else '✗':<10}")
    print("-" * 50)
    
    # Show which services will use stubs
    if use_stubs:
        print("\nServices using stubs:")
        for key, value in api_keys.items():
            if not value:
                print(f"  - {key}: Stub mode")
    
except Exception as e:
    print(f"❌ Configuration Error: {e}")
    print("Check your .env file format and API keys")


Configuration Loaded Successfully!
Tavily Enabled: True
Stub Mode: False

Settings Summary:
--------------------------------------------------
Setting              Value           Status    
--------------------------------------------------
OPENAI_API_KEY       ****            ✓         
TAVILY_API_KEY       ****            ✓         
COHERE_API_KEY       ****            ✓         
LANGCHAIN_API_KEY    ****            ✓         
USE_TAVILY           1               ✓         
USE_STUBS            0               ✗         
--------------------------------------------------


## Goal: Define minimal typed containers for docs and mcqs so agents stay compatible.


In [3]:
# Section 1: State Models - Typed Containers for Docs and MCQs
from typing import TypedDict, Literal, List, Dict, Any
import pandas as pd

# Document models
class Doc(TypedDict):
    """Standard document format from any source."""
    source: Literal["pubmed", "tavily"]
    id: str  # PMID for pubmed, unique ID for tavily
    title: str
    url: str
    text: str

class DocWithScore(TypedDict):
    """Document with relevance score for re-ranking."""
    source: Literal["pubmed", "tavily"]
    id: str
    title: str
    url: str
    text: str
    score: float

# MCQ model
class MCQ(TypedDict):
    """Multiple choice question with metadata."""
    stem: str
    options: List[str]  # 5 options
    answer_idx: int  # 0-4 index of correct answer
    rationale: str
    citations: List[Dict[str, Any]]  # List of citation dicts with pmid, quote, url

def pretty_docs(docs: List[Doc]) -> pd.DataFrame:
    """Display documents in a clean table format."""
    if not docs:
        return pd.DataFrame(columns=['Source', 'ID', 'Title', 'URL', 'Text'])
    
    data = []
    for doc in docs:
        # Truncate text for display
        text_preview = doc['text'][:100] + "..." if len(doc['text']) > 100 else doc['text']
        data.append({
            'Source': doc['source'],
            'ID': doc['id'],
            'Title': doc['title'][:50] + "..." if len(doc['title']) > 50 else doc['title'],
            'URL': doc['url'],
            'Text': text_preview
        })
    
    return pd.DataFrame(data)

# Test the models
print("State models imported successfully!")
print(f"Doc type: {Doc}")
print(f"DocWithScore type: {DocWithScore}")
print(f"MCQ type: {MCQ}")

# Test pretty_docs with empty list
empty_df = pretty_docs([])
print(f"\nEmpty docs DataFrame shape: {empty_df.shape}")
print("Empty DataFrame columns:", list(empty_df.columns))

# Test with sample data
sample_docs = [
    Doc(
        source="pubmed",
        id="12345678",
        title="Sample Medical Study",
        url="https://pubmed.ncbi.nlm.nih.gov/12345678/",
        text="This is a sample abstract about medical research findings."
    ),
    Doc(
        source="tavily",
        id="web_001",
        title="Clinical Guidelines",
        url="https://example.com/guidelines",
        text="These are clinical practice guidelines for medical professionals."
    )
]

sample_df = pretty_docs(sample_docs)
print(f"\nSample docs DataFrame shape: {sample_df.shape}")
print("\nSample DataFrame:")
print(sample_df.to_string(index=False))


State models imported successfully!
Doc type: <class '__main__.Doc'>
DocWithScore type: <class '__main__.DocWithScore'>
MCQ type: <class '__main__.MCQ'>

Empty docs DataFrame shape: (0, 5)
Empty DataFrame columns: ['Source', 'ID', 'Title', 'URL', 'Text']

Sample docs DataFrame shape: (2, 5)

Sample DataFrame:
Source       ID                Title                                       URL                                                              Text
pubmed 12345678 Sample Medical Study https://pubmed.ncbi.nlm.nih.gov/12345678/        This is a sample abstract about medical research findings.
tavily  web_001  Clinical Guidelines            https://example.com/guidelines These are clinical practice guidelines for medical professionals.


## Goal: Fetch PubMed docs only; tag source="pubmed" for clear visualisation.

**Note:** PubMedAgent uses E-utilities (esearch/efetch). In stub mode, return 3–5 hard-coded rows.


In [4]:
# Section 2: PubMedAgent - PubMed E-utilities Integration
import requests
import time
from typing import List
import xml.etree.ElementTree as ET

def compose_query(objective: str) -> str:
    """Convert learning objective to PubMed search query."""
    # Simple query composition - can be enhanced later
    return objective.replace(" ", " AND ")

def search_pubmed(query: str, n: int) -> List[Doc]:
    """
    Search PubMed using E-utilities API.
    
    Args:
        query: Search query string
        n: Number of results to return
        
    Returns:
        List of Doc objects with source="pubmed", or empty list if API fails
    """
    
    try:
        # Step 1: Search for PMIDs using esearch
        esearch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
        esearch_params = {
            'db': 'pubmed',
            'term': query,
            'retmax': n,
            'retmode': 'json',
            'sort': 'relevance'
        }
        
        print(f"Searching PubMed for: {query}")
        esearch_response = requests.get(esearch_url, params=esearch_params, timeout=10)
        esearch_response.raise_for_status()
        esearch_data = esearch_response.json()
        
        if 'esearchresult' not in esearch_data or not esearch_data['esearchresult']['idlist']:
            print("No results found in PubMed")
            return []
        
        pmids = esearch_data['esearchresult']['idlist']
        print(f"Found {len(pmids)} PMIDs")
        
        # Step 2: Fetch details using efetch
        efetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        efetch_params = {
            'db': 'pubmed',
            'id': ','.join(pmids),
            'retmode': 'xml'
        }
        
        efetch_response = requests.get(efetch_url, params=efetch_params, timeout=15)
        efetch_response.raise_for_status()
        
        # Parse XML response
        root = ET.fromstring(efetch_response.content)
        docs = []
        
        for article in root.findall('.//PubmedArticle'):
            try:
                # Extract PMID
                pmid_elem = article.find('.//PMID')
                pmid = pmid_elem.text if pmid_elem is not None else "unknown"
                
                # Extract title
                title_elem = article.find('.//ArticleTitle')
                title = title_elem.text if title_elem is not None else "No title"
                
                # Extract abstract
                abstract_elem = article.find('.//AbstractText')
                abstract = abstract_elem.text if abstract_elem is not None else "No abstract available"
                
                # Truncate abstract to ~3-5k chars
                if len(abstract) > 4000:
                    abstract = abstract[:4000] + "..."
                
                # Create Doc object
                doc = Doc(
                    source="pubmed",
                    id=pmid,
                    title=title,
                    url=f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                    text=abstract
                )
                docs.append(doc)
                
            except Exception as e:
                print(f"Error parsing article: {e}")
                continue
        
        print(f"Successfully retrieved {len(docs)} PubMed documents")
        return docs
        
    except requests.RequestException as e:
        print(f"PubMed API error: {e}")
        print("Returning empty results - API call failed")
        return []
    except Exception as e:
        print(f"Unexpected error: {e}")
        print("Returning empty results - parsing failed")
        return []

# Demo the PubMedAgent
print("Testing PubMedAgent...")
objective = "hypertension management in primary care"
query = compose_query(objective)
print(f"Composed query: {query}")

# Search PubMed
docs = search_pubmed(query, n=8)
print(f"\nRetrieved {len(docs)} documents")

# Display results
if docs:
    df = pretty_docs(docs)
    print(f"\nPubMed Results DataFrame shape: {df.shape}")
    print("\nPubMed Results:")
    print(df.to_string(index=False))
else:
    print("No documents retrieved - API call may have failed")


Testing PubMedAgent...
Composed query: hypertension AND management AND in AND primary AND care
Searching PubMed for: hypertension AND management AND in AND primary AND care
Found 8 PMIDs
Successfully retrieved 8 PubMed documents

Retrieved 8 documents

PubMed Results DataFrame shape: (8, 5)

PubMed Results:
Source       ID                                                 Title                                       URL                                                                                                    Text
pubmed 38582094 Global burden of 288 causes of death and life expe... https://pubmed.ncbi.nlm.nih.gov/38582094/ Regular, detailed reporting on population health by underlying cause of death is fundamental for pub...
pubmed 38762324 Global burden and strength of evidence for 88 risk... https://pubmed.ncbi.nlm.nih.gov/38762324/ Understanding the health consequences associated with exposure to risk factors is necessary to infor...
pubmed 34967848 Cancer Incidence, Mortality

## Section 3: TavilyAgent - Web Search Integration

**Explain:** TavilyAgent retrieves reputable web resources; in stub mode, returns 3–5 realistic placeholders.


In [5]:
# Section 3: TavilyAgent - Web Search Integration
import requests
from typing import List

def search_tavily(query: str, n: int) -> List[Doc]:
    """
    Search web using Tavily API.
    
    Args:
        query: Search query string
        n: Number of results to return
        
    Returns:
        List of Doc objects with source="tavily", or empty list if disabled/fails
    """
    
    # Check if Tavily is disabled
    if not use_tavily:
        print("Tavily search is disabled (USE_TAVILY=False)")
        return []
    
    # Check if we should use stubs
    if use_stubs:
        print("Stub mode enabled - Tavily API not called")
        print("To test Tavily search, set USE_STUBS=False in your .env file")
        return []
    
    # Check if API key is available
    if not api_keys['TAVILY_API_KEY']:
        print("Tavily API key not found")
        print("Add TAVILY_API_KEY to your .env file to enable web search")
        return []
    
    try:
        # Call Tavily Search API
        tavily_url = "https://api.tavily.com/search"
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_keys['TAVILY_API_KEY']}"
        }
        
        payload = {
            "query": query,
            "search_depth": "basic",
            "include_answer": False,
            "include_images": False,
            "include_raw_content": False,
            "max_results": n
        }
        
        print(f"Searching Tavily for: {query}")
        response = requests.post(tavily_url, headers=headers, json=payload, timeout=15)
        response.raise_for_status()
        
        data = response.json()
        
        if 'results' not in data or not data['results']:
            print("No results found in Tavily")
            return []
        
        docs = []
        for result in data['results']:
            try:
                # Extract information from Tavily result
                title = result.get('title', 'No title')
                url = result.get('url', '')
                content = result.get('content', '')
                
                # Use snippet if available, otherwise truncate content
                text = result.get('snippet', content)
                if len(text) > 1000:
                    text = text[:1000] + "..."
                
                # Create Doc object
                doc = Doc(
                    source="tavily",
                    id=url.split('/')[-1] if url else f"tavily_{len(docs)}",
                    title=title,
                    url=url,
                    text=text
                )
                docs.append(doc)
                
            except Exception as e:
                print(f"Error processing Tavily result: {e}")
                continue
        
        print(f"Successfully retrieved {len(docs)} Tavily documents")
        return docs
        
    except requests.RequestException as e:
        print(f"Tavily API error: {e}")
        print("Returning empty results - API call failed")
        return []
    except Exception as e:
        print(f"Unexpected error: {e}")
        print("Returning empty results - parsing failed")
        return []

# Demo the TavilyAgent
print("Testing TavilyAgent...")
query = "hypertension management guidelines"
print(f"Search query: {query}")

# Search Tavily
docs = search_tavily(query, n=8)
print(f"\nRetrieved {len(docs)} documents")

# Display results
if docs:
    df = pretty_docs(docs)
    print(f"\nTavily Results DataFrame shape: {df.shape}")
    print("\nTavily Results:")
    print(df.to_string(index=False))
else:
    print("No documents retrieved - check USE_TAVILY and TAVILY_API_KEY settings")


Testing TavilyAgent...
Search query: hypertension management guidelines
Searching Tavily for: hypertension management guidelines
Successfully retrieved 8 Tavily documents

Retrieved 8 documents

Tavily Results DataFrame shape: (8, 5)

Tavily Results:
Source                                          ID                                                 Title                                                                                                       URL                                                                                                    Text
tavily                   HYPERTENSIONAHA.120.15026 2020 International Society of Hypertension Global ...                                         https://www.ahajournals.org/doi/10.1161/HYPERTENSIONAHA.120.15026 The International Society of Hypertension (ISH) has developed worldwide practice guidelines for the ...
tavily    Elevated-Blood-Pressure-and-Hypertension ESC Guidelines for the management of elevated bloo... https://www.es

## Section 4: Merge + Re-rank - Combine and Rank Results

**Goal:** Concatenate outputs from both agents, then run Cohere Re-rank across the combined set.

**Note:** Re-ranking runs on the combined agent results. If USE_STUBS=True or COHERE_API_KEY missing, use a deterministic cosine-sim fallback for testing. Do return missing Cohere or search so user knows.


In [6]:
# Section 4: Merge + Re-rank - Combine and Rank Results
import requests
from typing import List
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def merge_results(pubmed_docs: List[Doc], tavily_docs: List[Doc]) -> List[Doc]:
    """
    Simple concatenation of PubMed and Tavily results.
    
    Args:
        pubmed_docs: List of PubMed documents
        tavily_docs: List of Tavily documents
        
    Returns:
        Combined list of all documents
    """
    combined = pubmed_docs + tavily_docs
    print(f"Merged {len(pubmed_docs)} PubMed + {len(tavily_docs)} Tavily = {len(combined)} total documents")
    return combined

def rerank_cohere(query: str, docs: List[Doc], top_k: int = 10) -> List[DocWithScore]:
    """
    Re-rank documents using Cohere API or fallback to cosine similarity.
    
    Args:
        query: Search query string
        docs: List of documents to re-rank
        top_k: Number of top results to return
        
    Returns:
        List of DocWithScore objects with relevance scores
    """
    
    if not docs:
        print("No documents to re-rank")
        return []
    
    # Check if we should use Cohere
    if use_stubs:
        print("Stub mode enabled - using cosine similarity fallback")
        return _rerank_cosine_similarity(query, docs, top_k)
    
    if not api_keys['COHERE_API_KEY']:
        print("Cohere API key not found - using cosine similarity fallback")
        print("Add COHERE_API_KEY to your .env file for better re-ranking")
        return _rerank_cosine_similarity(query, docs, top_k)
    
    try:
        # Call Cohere Re-rank API
        cohere_url = "https://api.cohere.ai/v1/rerank"
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_keys['COHERE_API_KEY']}"
        }
        
        # Prepare documents for Cohere
        documents = []
        for doc in docs:
            doc_text = f"{doc['title']} {doc['text']}"
            documents.append(doc_text)
        
        payload = {
            "model": "rerank-english-v3.0",
            "query": query,
            "documents": documents,
            "top_k": top_k,
            "return_documents": True
        }
        
        print(f"Re-ranking {len(docs)} documents using Cohere")
        response = requests.post(cohere_url, headers=headers, json=payload, timeout=15)
        response.raise_for_status()
        
        data = response.json()
        
        if 'results' not in data:
            print("No results from Cohere re-ranking")
            return _rerank_cosine_similarity(query, docs, top_k)
        
        # Convert Cohere results to DocWithScore
        ranked_docs = []
        for result in data['results']:
            try:
                doc_index = result['index']
                score = result['relevance_score']
                
                # Get original document
                original_doc = docs[doc_index]
                
                # Create DocWithScore
                doc_with_score = DocWithScore(
                    source=original_doc['source'],
                    id=original_doc['id'],
                    title=original_doc['title'],
                    url=original_doc['url'],
                    text=original_doc['text'],
                    score=score
                )
                ranked_docs.append(doc_with_score)
                
            except Exception as e:
                print(f"Error processing Cohere result: {e}")
                continue
        
        print(f"Successfully re-ranked {len(ranked_docs)} documents using Cohere")
        return ranked_docs
        
    except requests.RequestException as e:
        print(f"Cohere API error: {e}")
        print("Falling back to cosine similarity")
        return _rerank_cosine_similarity(query, docs, top_k)
    except Exception as e:
        print(f"Unexpected error: {e}")
        print("Falling back to cosine similarity")
        return _rerank_cosine_similarity(query, docs, top_k)

def _rerank_cosine_similarity(query: str, docs: List[Doc], top_k: int) -> List[DocWithScore]:
    """
    Fallback re-ranking using cosine similarity with TF-IDF.
    
    Args:
        query: Search query string
        docs: List of documents to re-rank
        top_k: Number of top results to return
        
    Returns:
        List of DocWithScore objects with cosine similarity scores
    """
    print("Using cosine similarity fallback for re-ranking")
    
    if not docs:
        return []
    
    try:
        # Prepare documents for TF-IDF
        documents = []
        for doc in docs:
            doc_text = f"{doc['title']} {doc['text']}"
            documents.append(doc_text)
        
        # Add query to documents for TF-IDF
        all_texts = [query] + documents
        
        # Compute TF-IDF
        vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
        tfidf_matrix = vectorizer.fit_transform(all_texts)
        
        # Compute cosine similarity between query and documents
        query_vector = tfidf_matrix[0:1]
        doc_vectors = tfidf_matrix[1:]
        
        similarities = cosine_similarity(query_vector, doc_vectors).flatten()
        
        # Create DocWithScore objects
        ranked_docs = []
        for i, (doc, score) in enumerate(zip(docs, similarities)):
            doc_with_score = DocWithScore(
                source=doc['source'],
                id=doc['id'],
                title=doc['title'],
                url=doc['url'],
                text=doc['text'],
                score=float(score)
            )
            ranked_docs.append(doc_with_score)
        
        # Sort by score (descending) and return top_k
        ranked_docs.sort(key=lambda x: x['score'], reverse=True)
        ranked_docs = ranked_docs[:top_k]
        
        print(f"Re-ranked {len(ranked_docs)} documents using cosine similarity")
        return ranked_docs
        
    except Exception as e:
        print(f"Error in cosine similarity fallback: {e}")
        print("Returning original documents without re-ranking")
        # Return original documents as DocWithScore with score 0
        return [DocWithScore(**doc, score=0.0) for doc in docs[:top_k]]

# Demo the merge and re-rank functionality
print("Testing Merge + Re-rank...")
query = "hypertension management guidelines"

# Get results from both agents
print("\n1. Getting PubMed results...")
pubmed_docs = search_pubmed(query, 8)
print(f"PubMed: {len(pubmed_docs)} documents")

print("\n2. Getting Tavily results...")
tavily_docs = search_tavily(query, 8)
print(f"Tavily: {len(tavily_docs)} documents")

# Merge results
print("\n3. Merging results...")
combined_docs = merge_results(pubmed_docs, tavily_docs)

# Re-rank results
print("\n4. Re-ranking results...")
ranked_docs = rerank_cohere(query, combined_docs, top_k=8)

# Display results
if ranked_docs:
    print(f"\n5. Top {len(ranked_docs)} ranked results:")
    print("-" * 80)
    print(f"{'Source':<10} {'Score':<8} {'Title':<50} {'URL':<20}")
    print("-" * 80)
    
    for doc in ranked_docs:
        title_short = doc['title'][:47] + "..." if len(doc['title']) > 50 else doc['title']
        url_short = doc['url'][:17] + "..." if len(doc['url']) > 20 else doc['url']
        print(f"{doc['source']:<10} {doc['score']:<8.3f} {title_short:<50} {url_short:<20}")
    
    print("-" * 80)
    
    # Show source distribution
    pubmed_count = sum(1 for doc in ranked_docs if doc['source'] == 'pubmed')
    tavily_count = sum(1 for doc in ranked_docs if doc['source'] == 'tavily')
    print(f"\nSource distribution: PubMed={pubmed_count}, Tavily={tavily_count}")
else:
    print("No documents to display")


Testing Merge + Re-rank...

1. Getting PubMed results...
Searching PubMed for: hypertension management guidelines
Found 8 PMIDs
Successfully retrieved 8 PubMed documents
PubMed: 8 documents

2. Getting Tavily results...
Searching Tavily for: hypertension management guidelines
Successfully retrieved 8 Tavily documents
Tavily: 8 documents

3. Merging results...
Merged 8 PubMed + 8 Tavily = 16 total documents

4. Re-ranking results...
Re-ranking 16 documents using Cohere
Successfully re-ranked 16 documents using Cohere

5. Top 16 ranked results:
--------------------------------------------------------------------------------
Source     Score    Title                                              URL                 
--------------------------------------------------------------------------------
tavily     0.998    Guideline-Driven Management of Hypertension: An... https://pmc.ncbi....
pubmed     0.997    The Japanese Society of Hypertension Guidelines... https://pubmed.nc...
tavily     0.

## Section 5: MCQ Generation - Create Questions from Ranked Results

**Explain:** This is a first-pass MCQ generator. It uses the top ranked docs as context. Keep constraints: 5 options, single best answer, no negation stems. Show correct option separately.


In [7]:
# Section 5: MCQ Generation - Create Questions from Ranked Results
from openai import OpenAI
import json
from typing import List, Dict, Any

def generate_mcqs(objective: Dict[str, Any], ranked_docs: List[DocWithScore]) -> List[MCQ]:
    """
    Generate MCQs using OpenAI GPT-4o-mini with ranked documents as context.
    
    Args:
        objective: Dict with 'n_mcq' key for number of questions to generate
        ranked_docs: List of ranked documents with scores
        
    Returns:
        List of MCQ objects, or empty list if API fails
    """
    
    if not ranked_docs:
        print("No ranked documents provided for MCQ generation")
        return []
    
    # Check if we should use stubs
    if use_stubs:
        print("Stub mode enabled - OpenAI API not called")
        print("To test MCQ generation, set USE_STUBS=False in your .env file")
        return []
    
    # Check if API key is available
    if not api_keys['OPENAI_API_KEY']:
        print("OpenAI API key not found")
        print("Add OPENAI_API_KEY to your .env file to enable MCQ generation")
        return []
    
    try:
        # Construct compact context from top 5 documents
        context_docs = ranked_docs[:5]
        context_parts = []
        
        for i, doc in enumerate(context_docs, 1):
            # Truncate text to keep context manageable
            text_snippet = doc['text'][:500] + "..." if len(doc['text']) > 500 else doc['text']
            context_parts.append(f"Document {i} ({doc['source']}): {doc['title']}\n{text_snippet}\nURL: {doc['url']}\n")
        
        context = "\n".join(context_parts)
        
        # Get number of MCQs to generate
        n_mcq = objective.get('n_mcq', 2)  # Default to 2 if not specified
        
        # Construct the prompt
        prompt = f"""You are a medical education expert creating high-quality multiple choice questions.

CONTEXT:
{context}

INSTRUCTIONS:
- Generate {n_mcq} MCQ(s) based on the provided context
- Each MCQ must have exactly 5 options (A, B, C, D, E)
- Ensure there is only ONE correct answer (single best answer)
- Do NOT use negation stems (avoid "Which of the following is NOT...")
- Provide brief clinical context if relevant
- Include proper citations with URLs

FORMAT:
For each MCQ, provide:
1. Clinical Context: [Brief relevant clinical scenario if applicable]
2. Stem: [The question]
3. Options: [A) option1, B) option2, C) option3, D) option4, E) option5]
4. Correct Answer: [The correct option letter]
5. Rationale: [Brief explanation]
6. Citations: [List of relevant URLs with brief quotes]

Return as JSON array with this structure:
[
  {{
    "stem": "question text",
    "options": ["option1", "option2", "option3", "option4", "option5"],
    "answer_idx": 0,
    "rationale": "explanation",
    "citations": [{{"url": "url1", "quote": "relevant quote"}}]
  }}
]
"""

        # Initialize OpenAI client
        client = OpenAI(api_key=api_keys['OPENAI_API_KEY'])
        
        print(f"Generating {n_mcq} MCQ(s) using OpenAI GPT-4o-mini...")
        
        # Call OpenAI API
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a medical education expert specializing in creating high-quality multiple choice questions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,  # Lower temperature for consistency
            max_tokens=2000
        )
        
        # Parse the response
        content = response.choices[0].message.content
        
        # Try to extract JSON from the response
        try:
            # Look for JSON array in the response
            start_idx = content.find('[')
            end_idx = content.rfind(']') + 1
            
            if start_idx != -1 and end_idx != 0:
                json_str = content[start_idx:end_idx]
                mcq_data = json.loads(json_str)
            else:
                print("No JSON array found in response")
                return []
            
            # Convert to MCQ objects
            mcqs = []
            for item in mcq_data:
                try:
                    mcq = MCQ(
                        stem=item['stem'],
                        options=item['options'],
                        answer_idx=item['answer_idx'],
                        rationale=item['rationale'],
                        citations=item['citations']
                    )
                    mcqs.append(mcq)
                except KeyError as e:
                    print(f"Missing key in MCQ data: {e}")
                    continue
            
            print(f"Successfully generated {len(mcqs)} MCQ(s)")
            return mcqs
            
        except json.JSONDecodeError as e:
            print(f"JSON parsing error: {e}")
            print("Raw response:", content[:200] + "...")
            return []
        
    except Exception as e:
        print(f"OpenAI API error: {e}")
        print("Returning empty results - API call failed")
        return []

def display_mcqs(mcqs: List[MCQ]) -> None:
    """Display MCQs in a readable format."""
    if not mcqs:
        print("No MCQs to display")
        return
    
    for i, mcq in enumerate(mcqs, 1):
        print(f"\n{'='*80}")
        print(f"MCQ {i}")
        print(f"{'='*80}")
        
        print(f"\nStem: {mcq['stem']}")
        
        print(f"\nOptions:")
        for j, option in enumerate(mcq['options']):
            letter = chr(65 + j)  # A, B, C, D, E
            print(f"  {letter}) {option}")
        
        correct_letter = chr(65 + mcq['answer_idx'])
        print(f"\nCorrect Answer: {correct_letter}) {mcq['options'][mcq['answer_idx']]}")
        
        print(f"\nRationale: {mcq['rationale']}")
        
        print(f"\nCitations:")
        for citation in mcq['citations']:
            print(f"  - {citation['url']}")
            print(f"    Quote: {citation['quote']}")
        
        print(f"\n{'-'*80}")

# Demo the MCQ generation
print("Testing MCQ Generation...")

# Create objective dict
objective = {
    "n_mcq": 2,
    "topic": "hypertension management"
}

# Use the ranked results from previous section
if 'ranked_docs' in locals() and ranked_docs:
    print(f"Using {len(ranked_docs)} ranked documents as context")
    
    # Generate MCQs
    mcqs = generate_mcqs(objective, ranked_docs)
    
    if mcqs:
        print(f"\nGenerated {len(mcqs)} MCQ(s)")
        display_mcqs(mcqs)
    else:
        print("No MCQs generated - check API configuration")
else:
    print("No ranked documents available - run the merge + re-rank section first")


Testing MCQ Generation...
Using 16 ranked documents as context
Generating 2 MCQ(s) using OpenAI GPT-4o-mini...
Successfully generated 2 MCQ(s)

Generated 2 MCQ(s)

MCQ 1

Stem: What is the recommended first-line treatment for adults with stage 1 hypertension according to the 2020 International Society of Hypertension guidelines?

Options:
  A) Lifestyle modifications only
  B) Thiazide diuretics
  C) ACE inhibitors
  D) Beta-blockers
  E) Calcium channel blockers

Correct Answer: B) Thiazide diuretics

Rationale: The 2020 International Society of Hypertension guidelines recommend thiazide diuretics as a first-line treatment for adults with stage 1 hypertension, especially in the presence of cardiovascular risk factors.

Citations:
  - https://www.ahajournals.org/doi/10.1161/HYPERTENSIONAHA.120.15026
    Quote: Thiazide diuretics are recommended as first-line treatment for stage 1 hypertension.

--------------------------------------------------------------------------------

MCQ 2

Ste

## Section 6: End-to-End Run - Complete Pipeline Execution

**Goal:** Single button-press cell that executes clarify → agents → merge → re-rank → MCQ generate.


In [8]:
# Section 6: End-to-End Run - Complete Pipeline Execution
from typing import Dict, Any

def clarify_objective() -> Dict[str, Any]:
    """
    Simple objective clarification - can be enhanced later with LangChain.
    
    Returns:
        Dict with objective details including n_mcq
    """
    # For now, return a simple objective
    # In future, this could be interactive or use LangChain for clarification
    objective = {
        "topic": "hypertension management in primary care",
        "n_mcq": 2,
        "description": "Generate MCQs about hypertension management approaches in primary care settings"
    }
    
    print(f"Objective clarified: {objective['description']}")
    return objective

def run_pipeline() -> List[MCQ]:
    """
    Execute the complete multiagent pipeline.
    
    Returns:
        List of generated MCQs, or empty list if pipeline fails
    """
    
    print("🚀 Starting Multiagent Pipeline...")
    print("=" * 60)
    
    try:
        # Step 1: Clarify objective
        print("\n1️⃣ Clarifying objective...")
        objective = clarify_objective()
        
        # Step 2: Build query
        print("\n2️⃣ Building search query...")
        query = compose_query(objective["topic"])
        print(f"Query: {query}")
        
        # Step 3: Search PubMed
        print("\n3️⃣ Searching PubMed...")
        pubmed_docs = search_pubmed(query, n=8)
        print(f"PubMed results: {len(pubmed_docs)} documents")
        
        # Step 4: Search Tavily
        print("\n4️⃣ Searching Tavily...")
        tavily_docs = search_tavily(query, n=8)
        print(f"Tavily results: {len(tavily_docs)} documents")
        
        # Step 5: Merge results
        print("\n5️⃣ Merging results...")
        combined_docs = merge_results(pubmed_docs, tavily_docs)
        
        if not combined_docs:
            print("❌ No documents found from either agent")
            return []
        
        # Step 6: Re-rank results
        print("\n6️⃣ Re-ranking results...")
        ranked_docs = rerank_cohere(query, combined_docs, top_k=8)
        
        if not ranked_docs:
            print("❌ Re-ranking failed")
            return []
        
        # Step 7: Generate MCQs
        print("\n7️⃣ Generating MCQs...")
        mcqs = generate_mcqs(objective, ranked_docs)
        
        if not mcqs:
            print("❌ MCQ generation failed")
            return []
        
        print(f"\n✅ Pipeline completed successfully!")
        print(f"Generated {len(mcqs)} MCQ(s)")
        
        return mcqs
        
    except Exception as e:
        print(f"\n❌ Pipeline failed with error: {e}")
        return []

def run_pipeline_with_summary() -> None:
    """
    Run the complete pipeline and display a summary of results.
    """
    
    # Execute pipeline
    mcqs = run_pipeline()
    
    if mcqs:
        print("\n" + "=" * 80)
        print("📊 PIPELINE SUMMARY")
        print("=" * 80)
        
        # Display MCQs
        display_mcqs(mcqs)
        
        # Summary statistics
        print(f"\n📈 Summary:")
        print(f"  • Total MCQs generated: {len(mcqs)}")
        print(f"  • Average options per MCQ: {sum(len(mcq['options']) for mcq in mcqs) / len(mcqs):.1f}")
        print(f"  • Total citations: {sum(len(mcq['citations']) for mcq in mcqs)}")
        
        # Check for different sources in citations
        all_citations = []
        for mcq in mcqs:
            all_citations.extend(mcq['citations'])
        
        pubmed_citations = sum(1 for citation in all_citations if 'pubmed' in citation['url'])
        tavily_citations = sum(1 for citation in all_citations if 'pubmed' not in citation['url'])
        
        print(f"  • PubMed citations: {pubmed_citations}")
        print(f"  • Web citations: {tavily_citations}")
        
        print(f"\n✅ End-to-end pipeline completed successfully!")
        
    else:
        print("\n❌ Pipeline failed - no MCQs generated")
        print("Check your API keys and configuration")

# Execute the complete pipeline
print("🎯 Running Complete Multiagent Pipeline...")
run_pipeline_with_summary()


🎯 Running Complete Multiagent Pipeline...
🚀 Starting Multiagent Pipeline...

1️⃣ Clarifying objective...
Objective clarified: Generate MCQs about hypertension management approaches in primary care settings

2️⃣ Building search query...
Query: hypertension AND management AND in AND primary AND care

3️⃣ Searching PubMed...
Searching PubMed for: hypertension AND management AND in AND primary AND care
Found 8 PMIDs
Successfully retrieved 8 PubMed documents
PubMed results: 8 documents

4️⃣ Searching Tavily...
Searching Tavily for: hypertension AND management AND in AND primary AND care
Successfully retrieved 8 Tavily documents
Tavily results: 8 documents

5️⃣ Merging results...
Merged 8 PubMed + 8 Tavily = 16 total documents

6️⃣ Re-ranking results...
Re-ranking 16 documents using Cohere
Successfully re-ranked 16 documents using Cohere

7️⃣ Generating MCQs...
Generating 2 MCQ(s) using OpenAI GPT-4o-mini...
Successfully generated 2 MCQ(s)

✅ Pipeline completed successfully!
Generated 2 MCQ(

## Section 7: Chunk and Store - Vector Database Integration

**Goal:** Store document chunks in embedded Qdrant for retrieval and similarity search.

**Note:** We use embedded Qdrant (no server) for quick local development. Collections: `pubmed_notes` and `notes`.


In [9]:
# Section 7: Chunk and Store - Vector Database Integration
import os
import uuid
from typing import List, Dict, Any, Optional
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Constants
COLL_PUBMED = "pubmed_notes"
COLL_NOTES = "notes"
VECTOR_SIZE = 1536  # OpenAI embedding size

def get_qdrant() -> QdrantClient:
    """Create and return an embedded Qdrant client."""
    try:
        client = QdrantClient(path="./qdrant_db")
        print("✅ Qdrant embedded client created successfully")
        return client
    except Exception as e:
        print(f"❌ Error creating Qdrant client: {e}")
        raise

def ensure_collections(client: QdrantClient, vector_size: int = VECTOR_SIZE) -> None:
    """Create collections if they don't exist."""
    try:
        # Check if collections exist
        collections = client.get_collections()
        existing_collections = [col.name for col in collections.collections]
        
        # Create pubmed_notes collection
        if COLL_PUBMED not in existing_collections:
            client.create_collection(
                collection_name=COLL_PUBMED,
                vectors_config=VectorParams(
                    size=vector_size,
                    distance=Distance.COSINE
                ),
                on_disk_payload=True
            )
            print(f"✅ Created collection: {COLL_PUBMED}")
        else:
            print(f"✅ Collection exists: {COLL_PUBMED}")
        
        # Create notes collection
        if COLL_NOTES not in existing_collections:
            client.create_collection(
                collection_name=COLL_NOTES,
                vectors_config=VectorParams(
                    size=vector_size,
                    distance=Distance.COSINE
                ),
                on_disk_payload=True
            )
            print(f"✅ Created collection: {COLL_NOTES}")
        else:
            print(f"✅ Collection exists: {COLL_NOTES}")
            
    except Exception as e:
        print(f"❌ Error creating collections: {e}")
        raise

def simple_chunk(text: str, max_tokens: int = 280) -> List[str]:
    """
    Simple text chunking with token approximation by character length.
    
    Args:
        text: Text to chunk
        max_tokens: Maximum tokens per chunk (approximated by char length)
        
    Returns:
        List of text chunks with overlap
    """
    try:
        # Approximate tokens by character length (rough estimate: 1 token ≈ 4 chars)
        max_chars = max_tokens * 4
        overlap_chars = 80  # ~60-80 chars overlap
        
        if len(text) <= max_chars:
            return [text]
        
        chunks = []
        start = 0
        
        while start < len(text):
            end = start + max_chars
            
            # If not the last chunk, try to break at sentence boundary
            if end < len(text):
                # Look for sentence endings within the last 100 chars
                for i in range(end, max(start + max_chars - 100, start), -1):
                    if text[i] in '.!?':
                        end = i + 1
                        break
            
            chunk = text[start:end].strip()
            if chunk:
                chunks.append(chunk)
            
            # Move start position with overlap
            start = end - overlap_chars
            if start >= len(text):
                break
        
        print(f"✅ Chunked text into {len(chunks)} pieces")
        return chunks
        
    except Exception as e:
        print(f"❌ Error chunking text: {e}")
        return [text]  # Return original text if chunking fails

def embed_texts(texts: List[str]) -> List[List[float]]:
    """
    Generate embeddings for texts using OpenAI.
    
    Args:
        texts: List of texts to embed
        
    Returns:
        List of embedding vectors
    """
    if not texts:
        return []
    
    # Check if we should use stubs
    if use_stubs:
        print("Stub mode enabled - OpenAI embeddings not called")
        print("To test embeddings, set USE_STUBS=False in your .env file")
        return []
    
    # Check if API key is available
    if not api_keys['OPENAI_API_KEY']:
        print("OpenAI API key not found")
        print("Add OPENAI_API_KEY to your .env file to enable embeddings")
        return []
    
    try:
        client = OpenAI(api_key=api_keys['OPENAI_API_KEY'])
        
        print(f"Generating embeddings for {len(texts)} texts...")
        
        # Call OpenAI embeddings API
        response = client.embeddings.create(
            model="text-embedding-3-small",  # Cost-effective embedding model
            input=texts
        )
        
        embeddings = [data.embedding for data in response.data]
        print(f"✅ Generated {len(embeddings)} embeddings")
        return embeddings
        
    except Exception as e:
        print(f"❌ OpenAI embeddings error: {e}")
        return []

def chunk_and_upsert_qdrant(docs: List[Doc], client: Optional[QdrantClient] = None) -> None:
    """
    Chunk documents and upsert to Qdrant with embeddings.
    
    Args:
        docs: List of documents to chunk and store
        client: Optional Qdrant client (will create if None)
    """
    if not docs:
        print("No documents to chunk and store")
        return
    
    try:
        # Get or create client
        if client is None:
            client = get_qdrant()
        
        # Ensure collections exist
        ensure_collections(client, VECTOR_SIZE)
        
        all_chunks = []
        all_payloads = []
        
        # Process each document
        for doc in docs:
            print(f"Processing {doc['source']} document: {doc['id']}")
            
            # Chunk the document text
            chunks = simple_chunk(doc['text'])
            
            # Create payloads for each chunk
            for i, chunk_text in enumerate(chunks):
                payload = {
                    "source": doc['source'],
                    "id": doc['id'],
                    "title": doc['title'],
                    "url": doc['url'],
                    "span_idx": i,
                    "text": chunk_text
                }
                all_payloads.append(payload)
                all_chunks.append(chunk_text)
        
        print(f"Total chunks created: {len(all_chunks)}")
        
        # Generate embeddings
        embeddings = embed_texts(all_chunks)
        
        if not embeddings:
            print("❌ No embeddings generated - cannot store chunks")
            return
        
        # Create points for Qdrant
        points = []
        for i, (payload, embedding) in enumerate(zip(all_payloads, embeddings)):
            # Create UUID from string ID
            point_id_str = f"{payload['source']}::{payload['id']}::{payload['span_idx']}"
            point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, point_id_str))
            
            point = PointStruct(
                id=point_id,
                vector=embedding,
                payload=payload
            )
            points.append(point)
        
        # Upsert to pubmed_notes collection
        client.upsert(
            collection_name=COLL_PUBMED,
            points=points
        )
        
        print(f"✅ Successfully stored {len(points)} chunks in {COLL_PUBMED}")
        
        # Verify storage with count query
        collection_info = client.get_collection(COLL_PUBMED)
        print(f"📊 Collection info: {collection_info.points_count} total points")
        
    except Exception as e:
        print(f"❌ Error in chunk_and_upsert_qdrant: {e}")
        raise

# Demo the chunking and storage system
print("Testing Chunk and Store System...")

# Check if client already exists and reuse it
if 'client' in globals() and client is not None:
    print("✅ Reusing existing Qdrant client")
else:
    # Get Qdrant client
    client = get_qdrant()


# Ensure collections exist
ensure_collections(client, VECTOR_SIZE)

# Use ranked documents from previous section if available
if 'ranked_docs' in locals() and ranked_docs:
    print(f"\nUsing {len(ranked_docs)} ranked documents for storage...")
    
    # Convert DocWithScore to Doc for storage
    docs_for_storage = []
    for doc in ranked_docs:
        doc_for_storage = Doc(
            source=doc['source'],
            id=doc['id'],
            title=doc['title'],
            url=doc['url'],
            text=doc['text']
        )
        docs_for_storage.append(doc_for_storage)
    
    # Chunk and store documents
    chunk_and_upsert_qdrant(docs_for_storage, client)
    
    print("\n✅ Chunk and store system test completed!")
else:
    print("No ranked documents available - run the merge + re-rank section first")


Testing Chunk and Store System...
✅ Qdrant embedded client created successfully
✅ Collection exists: pubmed_notes
✅ Collection exists: notes

Using 16 ranked documents for storage...
✅ Collection exists: pubmed_notes
✅ Collection exists: notes
Processing tavily document: 
Processing pubmed document: 31375757
Processing tavily document: 9789240033986
Processing tavily document: HYPERTENSIONAHA.120.15026
Processing tavily document: Elevated-Blood-Pressure-and-Hypertension
Processing pubmed document: 39970254
Processing pubmed document: 29133354
Processing pubmed document: 39210715
Processing pubmed document: 29133356
Processing pubmed document: 29146535
Processing tavily document: HYP.0000000000000249
Processing pubmed document: 32307541
Processing pubmed document: 38560900
Processing tavily document: 2023-ESH-Hypertension-Guideline-Update
Processing tavily document: hypertension-in-adults-initial-drug-therapy
Processing tavily document: p413.html
Total chunks created: 16
Generating embe

# Section 8: Mini Knowledge Graph - Regex-based Concept Extraction

## Goal: Create a tiny knowledge graph to guide MCQ stems and distractors

The KG is intentionally minimal—just enough to capture key medical concepts from abstracts. We'll use:
- **SQLite** for lightweight storage
- **Regex patterns** for sentence splitting and noun phrase extraction  
- **Medical keyword lists** for concept type classification
- **Evidence spans** (sentences) where concepts appear

This creates a deterministic, run-specific KG linked to your Qdrant-stored documents without NLTK dependencies.


In [10]:
# Section 8: Mini Knowledge Graph - Regex-based Concept Extraction

import sqlite3
import re
from typing import List, Tuple, Dict, Any

# Medical keyword categories for concept type classification
MEDICAL_KEYWORDS = {
    'risk_factor': ['risk', 'predictor', 'factor', 'associated', 'correlation', 'linked', 'related'],
    'intervention': ['treatment', 'therapy', 'drug', 'medication', 'intervention', 'management'],
    'outcome': ['outcome', 'result', 'mortality', 'survival', 'efficacy', 'response', 'improvement'],
    'mechanism': ['mechanism', 'pathway', 'inhibition', 'activation', 'process', 'function', 'action'],
    'finding': ['finding', 'study', 'trial', 'evidence', 'demonstrated', 'showed', 'revealed']
}

def get_kg(path: str = "./mini_kg.db") -> sqlite3.Connection:
    """Get SQLite connection to mini knowledge graph database."""
    try:
        conn = sqlite3.connect(path)
        return conn
    except Exception as e:
        print(f"Database connection error: {e}")
        return None

def init_schema(conn: sqlite3.Connection) -> None:
    """Initialize the mini knowledge graph schema."""
    try:
        cursor = conn.cursor()
        
        # Papers table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS papers (
                pmid TEXT PRIMARY KEY,
                title TEXT,
                year INTEGER,
                url TEXT
            )
        ''')
        
        # Concepts table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS concepts (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT,
                type TEXT
            )
        ''')
        
        # Paper-concepts linking table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS paper_concepts (
                pmid TEXT,
                concept_id INTEGER,
                evidence_span TEXT,
                FOREIGN KEY (pmid) REFERENCES papers(pmid),
                FOREIGN KEY (concept_id) REFERENCES concepts(id)
            )
        ''')
        
        # Learning objectives table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS learning_objectives (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                objective TEXT,
                audience TEXT,
                difficulty TEXT
            )
        ''')
        
        conn.commit()
        print("Mini KG schema initialized successfully")
        
    except Exception as e:
        print(f"Schema initialization error: {e}")
        conn.rollback()

def extract_concepts_regex(text: str) -> List[Tuple[str, str]]:
    """Extract medical concepts using regex patterns + keyword filtering."""
    try:
        # Split into sentences using regex
        sentences = re.split(r'[.!?]+', text)
        concepts = []
        
        for sentence in sentences:
            sentence = sentence.strip()
            if len(sentence) < 10:  # Skip very short sentences
                continue
                
            # Extract noun phrases using regex patterns
            # Pattern for common medical noun phrases
            noun_patterns = [
                r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b',  # Capitalized phrases
                r'\b(?:treatment|therapy|drug|medication|disease|condition|syndrome|disorder)\s+\w+\b',  # Medical terms + words
                r'\b\w+\s+(?:risk|factor|predictor|outcome|result|mechanism|pathway)\b',  # Words + medical keywords
                r'\b(?:blood|heart|lung|brain|liver|kidney)\s+\w+\b',  # Organ + word combinations
            ]
            
            noun_phrases = []
            for pattern in noun_patterns:
                matches = re.findall(pattern, sentence, re.IGNORECASE)
                noun_phrases.extend(matches)
            
            # Classify concepts by keyword matching
            for phrase in noun_phrases:
                phrase_lower = phrase.lower()
                for concept_type, keywords in MEDICAL_KEYWORDS.items():
                    if any(keyword in phrase_lower for keyword in keywords):
                        # Normalize concept name
                        normalized_name = re.sub(r'[^\w\s]', '', phrase).strip()
                        if normalized_name and len(normalized_name) > 2:
                            concepts.append((normalized_name, concept_type))
                            break  # Only assign one type per concept
        
        # Remove duplicates and limit to 5 per paper
        unique_concepts = list(set(concepts))
        return unique_concepts[:5]
        
    except Exception as e:
        print(f"Concept extraction error: {e}")
        return []

def kg_upsert_minimal(docs: List[Doc], objective: Dict[str, Any]) -> None:
    """Insert documents and extracted concepts into the mini knowledge graph."""
    try:
        conn = get_kg()
        if not conn:
            print("Cannot connect to database")
            return
            
        init_schema(conn)
        cursor = conn.cursor()
        
        # Insert learning objective
        cursor.execute('''
            INSERT INTO learning_objectives (objective, audience, difficulty)
            VALUES (?, ?, ?)
        ''', (objective.get('description', ''), 'medical_students', 'intermediate'))
        
        lo_id = cursor.lastrowid
        
        # Process each document
        for doc in docs:
            # Extract year from PubMed ID if available
            year = None
            if doc['source'] == 'pubmed' and doc['id'].isdigit():
                year = 2024  # Default for now
            
            # Insert paper
            cursor.execute('''
                INSERT OR REPLACE INTO papers (pmid, title, year, url)
                VALUES (?, ?, ?, ?)
            ''', (doc['id'], doc['title'], year, doc['url']))
            
            # Extract concepts using regex
            concepts = extract_concepts_regex(doc['text'])
            
            for concept_name, concept_type in concepts:
                # Insert concept
                cursor.execute('''
                    INSERT OR IGNORE INTO concepts (name, type)
                    VALUES (?, ?)
                ''', (concept_name, concept_type))
                
                # Get concept ID
                cursor.execute('SELECT id FROM concepts WHERE name = ? AND type = ?', 
                              (concept_name, concept_type))
                concept_id = cursor.fetchone()[0]
                
                # Link paper to concept with evidence span
                cursor.execute('''
                    INSERT INTO paper_concepts (pmid, concept_id, evidence_span)
                    VALUES (?, ?, ?)
                ''', (doc['id'], concept_id, doc['text'][:200] + "..."))
        
        conn.commit()
        print(f"Inserted {len(docs)} documents into mini KG")
        
    except Exception as e:
        print(f"KG upsert error: {e}")
        if conn:
            conn.rollback()

def create_mini_kg(docs: List[Doc], objective: Dict[str, Any]) -> None:
    """Main function to create mini knowledge graph from documents."""
    try:
        print("Creating mini knowledge graph...")
        kg_upsert_minimal(docs, objective)
        
        # Show preview of inserted data
        conn = get_kg()
        if conn:
            cursor = conn.cursor()
            cursor.execute('''
                SELECT p.title, c.name, c.type, pc.evidence_span
                FROM papers p
                JOIN paper_concepts pc ON p.pmid = pc.pmid
                JOIN concepts c ON pc.concept_id = c.id
                LIMIT 5
            ''')
            
            results = cursor.fetchall()
            if results:
                print("\nMini KG Preview (5 rows):")
                print("=" * 80)
                for title, concept, concept_type, evidence in results:
                    print(f"Paper: {title[:50]}...")
                    print(f"Concept: {concept} ({concept_type})")
                    print(f"Evidence: {evidence[:100]}...")
                    print("-" * 40)
            else:
                print("No data found in mini KG")
                
    except Exception as e:
        print(f"Mini KG creation error: {e}")

# Demo: Create mini KG from current top docs
print("Demo: Creating mini knowledge graph from current top documents...")

# Use the same docs from previous sections
if 'ranked_docs' in locals():
    create_mini_kg(ranked_docs, objective)
else:
    print("No ranked_docs available. Run previous sections first.")


Demo: Creating mini knowledge graph from current top documents...
Creating mini knowledge graph...
Mini KG schema initialized successfully
Inserted 16 documents into mini KG

Mini KG Preview (5 rows):
Paper: Hypertension in adults: Initial drug therapy - UpT...
Concept: ESH Guidelines for the management of arterial hypertension The Task Force for the management of arterial hypertension of the European (intervention)
Evidence: 2023 ESH Guidelines for the management of arterial hypertension The Task Force for the management of...
----------------------------------------
Paper: 2020 International Society of Hypertension Global ...
Concept: Clinical Practice Guidelines for the Management of Hypertension in the Community (intervention)
Evidence: To align with its mission to reduce the global burden of raised blood pressure (BP), the Internation...
----------------------------------------
Paper: 2020 International Society of Hypertension Global ...
Concept: has developed worldwide practice g

# Section 9: Retrieval Integration - Semantic + Knowledge Graph

## Goal: Merge semantic hits from Qdrant with KG-related papers

Retrieval merges semantic hits from Qdrant with KG-related papers; we enforce a token budget and deduplicate by PMID/URL.

**Key Features:**
- **Qdrant semantic search** using OpenAI embeddings
- **Knowledge Graph retrieval** via concept-paper links
- **Context merging** with 2:1 interleaving (Qdrant:KG)
- **Token budget** enforcement (~6000 characters)
- **Fallback handling** for failed retrievals


In [11]:
# Section 9: Retrieval Integration - Semantic + Knowledge Graph

import sqlite3
from typing import List, Dict, Any
from openai import OpenAI

def retrieve_qdrant(query: str, k: int = 6) -> List[Doc]:
    """Embed query, search COLL_PUBMED, reconstruct Doc objects (chunk text as text)."""
    try:
        # Reuse existing Qdrant client if available, otherwise create new one
        global client
        try:
            _ = client
        except NameError:
            client = get_qdrant()  #get_qdrant defined in section 7: chunking

        if not client:
            print("Qdrant client not available")
            return []
        
        # Embed query using OpenAI
        if not api_keys['OPENAI_API_KEY']:
            print("OpenAI API key not available for embedding")
            return []
            
        openai_client = OpenAI(api_key=api_keys['OPENAI_API_KEY'])
        
        # Generate query embedding
        query_embedding = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=query
        ).data[0].embedding
        
        # Search Qdrant
        search_results = client.search(
            collection_name=COLL_PUBMED,
            query_vector=query_embedding,
            limit=k
        )
        
        # Reconstruct Doc objects
        docs = []
        for result in search_results:
            payload = result.payload
            doc = Doc(
                source=payload.get('source', 'pubmed'),
                id=payload.get('id', ''),
                title=payload.get('title', ''),
                url=payload.get('url', ''),
                text=payload.get('text', '')  # Full chunk content
            )
            docs.append(doc)
        
        print(f"Retrieved {len(docs)} documents from Qdrant")
        return docs
        
    except Exception as e:
        print(f"Qdrant retrieval error: {e}")
        return []

def retrieve_kg(objective: dict, k: int = 4) -> List[Doc]:
    """Query mini-KG for top concepts linked to current learning objective, return linked papers."""
    try:
        # Get KG connection
        conn = get_kg()
        if not conn:
            print("KG database not available")
            return []
        
        # Extract objective text with fallback handling
        objective_text = objective.get('description', objective.get('topic', ''))
        
        # Skip KG lookup if no valid objective text
        if not objective_text or objective_text.strip() == '':
            print("No valid learning objective — skipping KG lookup.")
            return []
        
        cursor = conn.cursor()
        
        # Find learning objective
        cursor.execute('''
            SELECT id FROM learning_objectives 
            WHERE objective LIKE ? 
            ORDER BY id DESC LIMIT 1
        ''', (f'%{objective_text[:50]}%',))
        
        lo_result = cursor.fetchone()
        if not lo_result:
            print("No matching learning objective found in KG")
            return []
        
        # Find top-linked concepts and their papers
        cursor.execute('''
            SELECT p.pmid, p.title, p.url, COUNT(pc.concept_id) as link_count
            FROM papers p
            JOIN paper_concepts pc ON p.pmid = pc.pmid
            JOIN concepts c ON pc.concept_id = c.id
            GROUP BY p.pmid, p.title, p.url
            ORDER BY link_count DESC
            LIMIT ?
        ''', (k,))
        
        results = cursor.fetchall()
        
        # Convert to Doc objects
        docs = []
        for pmid, title, url, link_count in results:
            doc = Doc(
                source='pubmed',
                id=str(pmid),
                title=title or '',
                url=url or f'https://pubmed.ncbi.nlm.nih.gov/{pmid}/',
                text=''  # Empty text as specified
            )
            docs.append(doc)
        
        print(f"Retrieved {len(docs)} documents from KG")
        return docs
        
    except Exception as e:
        print(f"KG retrieval error: {e}")
        return []

def merge_context(q_hits: List[Doc], kg_hits: List[Doc], max_chars: int = 6000) -> List[Doc]:
    """Deduplicate by (source,id), interleave 2:1 favouring Qdrant, truncate total text ≤ max_chars."""
    try:
        # Deduplicate by (source, id) tuple
        seen = set()
        deduplicated_q = []
        deduplicated_kg = []
        
        for doc in q_hits:
            key = (doc['source'], doc['id'])
            if key not in seen:
                seen.add(key)
                deduplicated_q.append(doc)
        
        for doc in kg_hits:
            key = (doc['source'], doc['id'])
            if key not in seen:
                seen.add(key)
                deduplicated_kg.append(doc)
        
        # Interleave 2:1 (Qdrant:KG)
        merged = []
        q_idx = 0
        kg_idx = 0
        total_chars = 0
        
        while (q_idx < len(deduplicated_q) or kg_idx < len(deduplicated_kg)) and total_chars < max_chars:
            # Add 2 Qdrant hits
            for _ in range(2):
                if q_idx < len(deduplicated_q):
                    doc = deduplicated_q[q_idx]
                    if total_chars + len(doc['text']) <= max_chars:
                        merged.append(doc)
                        total_chars += len(doc['text'])
                    q_idx += 1
                else:
                    break
            
            # Add 1 KG hit
            if kg_idx < len(deduplicated_kg):
                doc = deduplicated_kg[kg_idx]
                if total_chars + len(doc['text']) <= max_chars:
                    merged.append(doc)
                    total_chars += len(doc['text'])
                kg_idx += 1
        
        print(f"Merged {len(merged)} documents, {total_chars} characters total")
        return merged
        
    except Exception as e:
        print(f"Context merging error: {e}")
        return q_hits + kg_hits  # Fallback to simple concatenation

def fallback_retrieval(query: str) -> List[Doc]:
    """Fallback LLM retrieval when Qdrant fails."""
    try:
        if not api_keys['OPENAI_API_KEY']:
            print("No OpenAI API key for fallback")
            return []
        
        openai_client = OpenAI(api_key=api_keys['OPENAI_API_KEY'])
        
        prompt = f"""
        Generate 3 short context paragraphs (200-300 words each) about: {query}
        
        Focus on medical/clinical information relevant to the query.
        Each paragraph should be informative and factual.
        """
        
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000
        )
        
        content = response.choices[0].message.content
        
        # Split into paragraphs and create Doc objects
        paragraphs = [p.strip() for p in content.split('\n\n') if p.strip()]
        
        docs = []
        for i, para in enumerate(paragraphs[:3]):
            doc = Doc(
                source='llm_fallback',
                id=f'fallback_{i}',
                title=f'LLM Fallback Context {i+1}',
                url='',
                text=para
            )
            docs.append(doc)
        
        print(f"Generated {len(docs)} fallback documents")
        return docs
        
    except Exception as e:
        print(f"Fallback retrieval error: {e}")
        return []

# Demo: Test retrieval integration with enhanced fallback handling
print("Demo: Testing retrieval integration...")

# Use objective and query from previous sections
if 'objective' in locals() and 'query' in locals():
    # Safe objective display with fallback
    objective_desc = objective.get('description', objective.get('topic', 'Unknown'))
    print(f"Objective: {objective_desc}")
    print(f"Query: {query}")
    
    # Test Qdrant retrieval
    print("\n1. Testing Qdrant retrieval...")
    qdrant_docs = retrieve_qdrant(query, k=6)
    
    # Test KG retrieval
    print("\n2. Testing KG retrieval...")
    kg_docs = retrieve_kg(objective, k=4)
    
    # Enhanced fallback handling
    if not qdrant_docs and not kg_docs:
        print("\n3. Both Qdrant and KG failed - triggering LLM fallback...")
        fallback_docs = fallback_retrieval(query)
        merged_docs = fallback_docs
    else:
        # Test context merging
        print("\n3. Testing context merging...")
        merged_docs = merge_context(qdrant_docs, kg_docs, max_chars=6000)
    
    # Display results
    if merged_docs:
        print(f"\nFinal merged context ({len(merged_docs)} documents):")
        print("=" * 80)
        print(f"{'Source':<15} {'ID':<15} {'Title':<40} {'URL':<20}")
        print("=" * 80)
        
        total_chars = 0
        for doc in merged_docs:
            title_short = doc['title'][:37] + "..." if len(doc['title']) > 40 else doc['title']
            url_short = doc['url'][:17] + "..." if len(doc['url']) > 20 else doc['url']
            print(f"{doc['source']:<15} {doc['id']:<15} {title_short:<40} {url_short:<20}")
            total_chars += len(doc['text'])
        
        print("=" * 80)
        print(f"Total characters: {total_chars}")
    else:
        print("No documents retrieved")
        
else:
    print("No objective or query available. Run previous sections first.")


Demo: Testing retrieval integration...
Objective: hypertension management
Query: hypertension management guidelines

1. Testing Qdrant retrieval...
Retrieved 6 documents from Qdrant

2. Testing KG retrieval...
Retrieved 4 documents from KG

3. Testing context merging...
Merged 9 documents, 3529 characters total

Final merged context (9 documents):
Source          ID              Title                                    URL                 
tavily          HYPERTENSIONAHA.120.15026 2020 International Society of Hyperte... https://www.ahajo...
pubmed          39970254        What Is New and Different in the 2024... https://pubmed.nc...
pubmed                          Guideline-Driven Management of Hypert... https://pmc.ncbi....
tavily          hypertension-in-adults-initial-drug-therapy Hypertension in adults: Initial drug ... https://www.uptod...
tavily          HYP.0000000000000249 2025 AHA/ACC/AANP/AAPA/ABC/ACCP/ACPM/... https://www.ahajo...
pubmed          changes-you-can-make-to-man

C:\Users\mspla\AppData\Local\Temp\ipykernel_17464\2003592004.py:35: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(


## Section 10: Hierarchical Orchestration with LangGraph (Supervisor + Researchers)

**Purpose:** Build a minimal supervisor graph that delegates to separate PubMed/Tavily researchers, merges & re-ranks, stores to Qdrant + mini-KG, retrieves context (Qdrant + KG) with LLM fallback, generates MCQs, and validates outputs.

**State keys:** objective, query, pub, web, ranked, context, mcqs, validation, used_fallback.

**Flow:** clarify → researchers → rank → store → retrieve → (router) → mcq → validate → END.

**Notes:**
- Use compose_query(objective["topic"]).
- Do not instantiate Doc/DocWithScore as classes; build plain dicts.
- If both Qdrant+KG empty, fallback generates 2–3 short paragraphs.
- MCQs must print at the end; raise if none.


In [12]:
# Section 10: Hierarchical Orchestration with LangGraph (Supervisor + Researchers)
from typing import TypedDict, List, Dict, Any
try:
    from langgraph.graph import StateGraph, START, END
except Exception:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "langgraph"], check=True)
    from langgraph.graph import StateGraph, END

# ------------------------
# State
# ------------------------
class OrchestratorState(TypedDict, total=False):
    objective: Dict[str, Any]
    query: str
    pub: List[Dict[str, Any]]
    web: List[Dict[str, Any]]
    ranked: List[Dict[str, Any]]
    context: List[Dict[str, Any]]
    mcqs: List[Dict[str, Any]]
    validation: Dict[str, Any]
    used_fallback: bool
    rubric: Dict[str, Any]  #used in section 11

# ------------------------
# Fallback (reuse your existing OpenAI client setup if present)
# ------------------------
def llm_fallback_context(topic: str) -> List[str]:
    """Return 2–3 short paragraphs if retrieval fails; stub if no key."""
    try:
        # Reuse your existing client/config if available
        if 'OPENAI_API_KEY' in api_keys and api_keys.get('OPENAI_API_KEY'):
            # Use the same client you use in MCQ generation
            # Expect a helper like openai_client already configured; if not, import your setup.
            from openai import OpenAI
            client = OpenAI(api_key=api_keys['OPENAI_API_KEY'])
            msg = [
                {"role": "system", "content": "You produce concise, factual medical teaching notes."},
                {"role": "user", "content": f"Write 3 short teaching paragraphs (120–160 words each) about: {topic}. Return plain text paragraphs separated by blank lines."}
            ]
            resp = client.chat.completions.create(model="gpt-4o-mini", messages=msg, max_tokens=600)
            content = resp.choices[0].message.content or ""
            paras = [p.strip() for p in content.split("\n\n") if p.strip()]
            return paras[:3] if paras else []
    except Exception as e:
        print("[fallback] LLM error:", e)
    # Stub fallback to keep pipeline running
    return [f"{topic}: overview paragraph (stub) #{i+1}." for i in range(3)]

# ------------------------
# Nodes (wrapping existing functions — keep names/signatures as implemented earlier)
# ------------------------
def clarify_node(state: OrchestratorState) -> OrchestratorState:
    obj = state.get("objective") or clarify_objective()
    query = compose_query(obj["topic"])
    print(f"✅ Objective: {obj['topic']}")
    return {**state, "objective": obj, "query": query}

def researchers_node(state: OrchestratorState) -> OrchestratorState:
    q = state["query"]
    pub = search_pubmed(q, n=8)
    web = search_tavily(q, n=8)  # function should honour USE_TAVILY flag internally
    print(f"🔎 PubMed={len(pub)} Tavily={len(web)}")
    return {**state, "pub": pub, "web": web}

def rank_node(state: OrchestratorState) -> OrchestratorState:
    q = state["query"]
    combined = merge_results(state.get("pub", []), state.get("web", []))
    ranked = rerank_cohere(q, combined, top_k=8)
    print(f"📈 Ranked={len(ranked)}")
    return {**state, "ranked": ranked[:8]}

def store_node(state: OrchestratorState) -> OrchestratorState:
    ranked = state.get("ranked", [])
    if not ranked:
        return state
    # Convert ranked (DocWithScore) → Doc dicts for storage
    docs_for_storage = [{"source": d["source"], "id": d["id"], "title": d["title"], "url": d.get("url",""), "text": d["text"]} for d in ranked]
    try:
        if 'client' in globals() and client is not None:
            chunk_and_upsert_qdrant(docs_for_storage, client)
        else:
            chunk_and_upsert_qdrant(docs_for_storage)
    except Exception as e:
        print("[store_node] Qdrant store warning:", e)
    try:
        kg_upsert_minimal(docs_for_storage, state["objective"])
    except Exception as e:
        print("[store_node] KG upsert warning:", e)
    return state

def retrieve_node(state: OrchestratorState) -> OrchestratorState:
    q = state["query"]
    obj = state["objective"]
    try:
        q_hits = retrieve_qdrant(q, k=6)
    except Exception as e:
        print("[retrieve_node] Qdrant error:", e)
        q_hits = []
    try:
        kg_hits = retrieve_kg(obj, k=4)
    except Exception as e:
        print("[retrieve_node] KG error:", e)
        kg_hits = []
    context = merge_context(q_hits, kg_hits, max_chars=6000)
    print(f"📚 Context={len(context)}")
    return {**state, "context": context}

def context_router(state: OrchestratorState) -> str:
    return "mcq_node" if state.get("context") else "fallback_node"

def fallback_node(state: OrchestratorState) -> OrchestratorState:
    topic = state["objective"]["topic"]
    paras = llm_fallback_context(topic)
    docs = [{"source": "llm_fallback", "id": f"fb-{i}", "title": f"LLM Fallback {i+1}", "url": "", "text": p} for i, p in enumerate(paras)]
    print(f"🧰 Fallback docs={len(docs)}")
    return {**state, "context": docs, "used_fallback": True}

def mcq_node(state: OrchestratorState) -> OrchestratorState:
    ctx = state.get("context", [])
    # Adapter: context Doc → DocWithScore (score=1.0)
    ranked_like = [{**d, "score": 1.0} for d in ctx]
    mcqs = generate_mcqs(state["objective"], ranked_like)
    if not mcqs:
        raise RuntimeError("MCQ generation returned no items. Check API key / USE_STUBS / context size.")
    print(f"🎯 MCQs={len(mcqs)}")
    return {**state, "mcqs": mcqs}

def validate_node(state: OrchestratorState) -> OrchestratorState:
    try:
        report = validate_mcqs(state["mcqs"], state["context"])
    except NameError:
        report = {"status": "skipped"}
    except Exception as e:
        report = {"status": "error", "error": str(e)}
    return {**state, "validation": report}

# ------------------------
# Graph wiring
# ------------------------
print("🏗️ Building LangGraph…")
g = StateGraph(OrchestratorState)
g.add_node("clarify_node", clarify_node)
g.add_node("researchers_node", researchers_node)
g.add_node("rank_node", rank_node)
g.add_node("store_node", store_node)
g.add_node("retrieve_node", retrieve_node)
g.add_node("fallback_node", fallback_node)
g.add_node("mcq_node", mcq_node)
g.add_node("validate_node", validate_node)

g.set_entry_point("clarify_node")
g.add_edge("clarify_node", "researchers_node")
g.add_edge("researchers_node", "rank_node")
g.add_edge("rank_node", "store_node")
g.add_edge("store_node", "retrieve_node")
g.add_conditional_edges("retrieve_node", context_router, {"mcq_node": "mcq_node", "fallback_node": "fallback_node"})
g.add_edge("fallback_node", "mcq_node")
g.add_edge("mcq_node", "validate_node")
g.add_edge("validate_node", END)
app = g.compile()
print("✅ LangGraph ready.")

# ------------------------
# Run once
# ------------------------
final_state = app.invoke({})
summary = {
    "pub": len(final_state.get("pub", [])),
    "web": len(final_state.get("web", [])),
    "ranked": len(final_state.get("ranked", [])),
    "context": len(final_state.get("context", [])),
    "mcqs": len(final_state.get("mcqs", [])),
    "used_fallback": final_state.get("used_fallback", False),
    "validation": final_state.get("validation", {}).get("status", "unknown")
}
print("\n📊 SUMMARY:", summary)

mcqs = final_state.get("mcqs", [])
if not mcqs:
    raise RuntimeError("❌ No MCQs generated — inspect logs above.")
print(f"\n=== GENERATED {len(mcqs)} MCQ(S) ===")
for i, q in enumerate(mcqs, 1):
    print(f"\nMCQ {i}: {q['stem']}")
    for j, opt in enumerate(q["options"]):
        print(f"  {chr(65+j)}. {opt}")
    print(f"  ✅ Answer: {chr(65 + q['answer_idx'])}")


🏗️ Building LangGraph…
✅ LangGraph ready.
Objective clarified: Generate MCQs about hypertension management approaches in primary care settings
✅ Objective: hypertension management in primary care
Searching PubMed for: hypertension AND management AND in AND primary AND care
Found 8 PMIDs
Successfully retrieved 8 PubMed documents
Searching Tavily for: hypertension AND management AND in AND primary AND care
Successfully retrieved 8 Tavily documents
🔎 PubMed=8 Tavily=8
Merged 8 PubMed + 8 Tavily = 16 total documents
Re-ranking 16 documents using Cohere
Successfully re-ranked 16 documents using Cohere
📈 Ranked=16
✅ Collection exists: pubmed_notes
✅ Collection exists: notes
Processing tavily document: 
Processing tavily document: 
Processing tavily document: CIRCRESAHA.121.318083
Processing tavily document: 
Processing tavily document: changes-you-can-make-to-manage-high-blood-pressure
Processing tavily document: hypertension-in-adults-initial-drug-therapy
Processing tavily document: 2833283

C:\Users\mspla\AppData\Local\Temp\ipykernel_17464\2003592004.py:35: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(


Retrieved 6 documents from Qdrant
Retrieved 4 documents from KG
Merged 10 documents, 2559 characters total
📚 Context=10
Generating 2 MCQ(s) using OpenAI GPT-4o-mini...
Successfully generated 2 MCQ(s)
🎯 MCQs=2

📊 SUMMARY: {'pub': 8, 'web': 8, 'ranked': 8, 'context': 10, 'mcqs': 2, 'used_fallback': False, 'validation': 'skipped'}

=== GENERATED 2 MCQ(S) ===

MCQ 1: In managing hypertension in primary care, which class of medication is recommended as first-line therapy for self-identified Black patients based on evidence from randomized trials?
  A. ACE inhibitors
  B. Beta-blockers
  C. Calcium channel blockers
  D. Thiazide-like diuretics
  E. Angiotensin II receptor blockers
  ✅ Answer: C

MCQ 2: What is a key recommendation from the 2025 AHA/ACC guidelines regarding the management of high blood pressure in adults?
  A. Use of monotherapy for all patients
  B. Routine use of beta-blockers as first-line therapy
  C. Incorporation of lifestyle modifications alongside pharmacotherapy
  D.

## Section 11: MCQ Rubric & Checks (Agent Node)

**Purpose:** Add a rubric evaluator agent node that scores each generated MCQ on:
- **Clarity** (no "EXCEPT/NOT"; stem ≤ 280 chars)
- **Relevance** (overlap with objective/query or retrieved context)
- **Distractor quality** (options unique; distractors not near-duplicates of the key)
- **Rationale match** (rationale mentions key concept more than distractors)
- **Single correct key** (exactly one answer index 0–4)

**Output:** `state['rubric'] = {'per_item': [...], 'overall': {...}}`

**Implementation:** Rebuild the entire LangGraph from Section 10 with the rubric_node inserted between mcq_node and validate_node. This ensures a clean single-path structure without concurrent update conflicts.

**Goal:** Keep pipeline deterministic: if retrieval falls back to LLM, rubric still runs and prints a summary.


In [13]:
# Section 11: MCQ Rubric & Checks (Agent Node)
from typing import Dict, Any, List
from langgraph.graph import START, END

# 1) Lightweight helpers (no heavy deps)
def _norm_text(s: str) -> str:
    return " ".join((s or "").lower().split())

def _token_set(s: str) -> set:
    return set(_norm_text(s).split())

def _jaccard(a: str, b: str) -> float:
    A, B = _token_set(a), _token_set(b)
    if not A or not B: 
        return 0.0
    return len(A & B) / len(A | B)

def _has_negation(s: str) -> bool:
    t = " " + _norm_text(s) + " "
    for bad in (" except ", " not ", " least likely ", " all of the following except "):
        if bad in t:
            return True
    return False

# 2) Rubric evaluator node (rule-based; runs without network)
def rubric_evaluator_node(state: OrchestratorState) -> OrchestratorState:
    mcqs: List[Dict[str, Any]] = state.get("mcqs", []) or []
    context_docs: List[Dict[str, Any]] = state.get("context", []) or []
    objective = state.get("objective", {}) or {}
    topic = objective.get("topic", "")
    context_blob = " ".join((d.get("text","") or "")[:1000] for d in context_docs)

    per_item = []
    for q in mcqs:
        stem = q.get("stem","")
        options = q.get("options", []) or []
        key_idx = q.get("answer_idx", -1)
        rationale = q.get("rationale","")

        single_key = int(0 <= key_idx < 5 and len(options) == 5)
        clarity = int((len(stem) <= 280) and (not _has_negation(stem)))
        rel_obj = _jaccard(stem, topic) >= 0.08
        rel_ctx = _jaccard(stem, context_blob) >= 0.05
        relevance = int(rel_obj or rel_ctx)

        uniq = len(set(_norm_text(o) for o in options)) == 5
        if uniq and 0 <= key_idx < 5:
            key_text = _norm_text(options[key_idx])
            distractors = [_norm_text(options[i]) for i in range(5) if i != key_idx]
            sim = [_jaccard(key_text, d) for d in distractors]
            dist_quality = int(all(s < 0.5 for s in sim))
        else:
            dist_quality = 0

        if rationale and 0 <= key_idx < 5:
            key_text = _norm_text(options[key_idx])
            rat_text = _norm_text(rationale)
            key_overlap = _jaccard(key_text, rat_text)
            distractors = [_norm_text(options[i]) for i in range(5) if i != key_idx]
            dist_overlaps = [_jaccard(d, rat_text) for d in distractors]
            rationale_match = int(all(key_overlap > do for do in dist_overlaps))
        else:
            rationale_match = 0

        score = single_key + clarity + relevance + dist_quality + rationale_match
        per_item.append({
            "stem": stem[:60] + "..." if len(stem) > 60 else stem,
            "single_key": single_key,
            "clarity": clarity,
            "relevance": relevance,
            "distractor_quality": dist_quality,
            "rationale_match": rationale_match,
            "total": score
        })

    avg_score = sum(it["total"] for it in per_item) / len(per_item) if per_item else 0.0
    overall = {
        "count": len(per_item),
        "avg_score": round(avg_score, 2),
        "max_score": 5
    }

    print(f"📊 Rubric: {len(per_item)} MCQs evaluated, avg={overall['avg_score']}/5")
    return {**state, "rubric": {"per_item": per_item, "overall": overall}}

# 3) Rebuild entire graph from Section 10 with rubric_node included
print("🔧 Rebuilding LangGraph with rubric_node...")

# Create fresh graph
workflow = StateGraph(OrchestratorState)

# Add all nodes (including rubric)
workflow.add_node("clarify", clarify_node)
workflow.add_node("researchers", researchers_node)
workflow.add_node("rank", rank_node)
workflow.add_node("store", store_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("mcq", mcq_node)
workflow.add_node("rubric", rubric_evaluator_node)  # NEW NODE
workflow.add_node("validate", validate_node)

# Add edges (single path: clarify → researchers → rank → store → retrieve → mcq → rubric → validate → END)
workflow.add_edge(START, "clarify")
workflow.add_edge("clarify", "researchers")
workflow.add_edge("researchers", "rank")
workflow.add_edge("rank", "store")
workflow.add_edge("store", "retrieve")
workflow.add_edge("retrieve", "mcq")
workflow.add_edge("mcq", "rubric")  # Insert rubric between mcq and validate
workflow.add_edge("rubric", "validate")  # Single path to validate
workflow.add_edge("validate", END)

# Compile fresh graph
app = workflow.compile()

print("✅ Graph rebuilt with rubric_node successfully!")

# 4) Demo run
print("\n" + "="*80)
print("Running pipeline with rubric evaluation...")
print("="*80)

objective_input = {
    "objective": {
        "topic": "hypertension management in primary care",
        "n_mcq": 2
    }
}

final_state = app.invoke(objective_input)

# Display rubric results
if "rubric" in final_state:
    rubric = final_state["rubric"]
    print("\n" + "="*80)
    print("RUBRIC EVALUATION RESULTS")
    print("="*80)
    
    overall = rubric.get("overall", {})
    print(f"\n📊 Overall: {overall.get('count', 0)} MCQs, "
          f"Avg Score: {overall.get('avg_score', 0)}/{overall.get('max_score', 5)}")
    
    print("\n📋 Per-Item Scores:")
    print("-" * 80)
    for i, item in enumerate(rubric.get("per_item", []), 1):
        print(f"\nMCQ {i}: {item.get('stem', 'N/A')}")
        print(f"  Single Key: {item.get('single_key', 0)}/1")
        print(f"  Clarity: {item.get('clarity', 0)}/1")
        print(f"  Relevance: {item.get('relevance', 0)}/1")
        print(f"  Distractor Quality: {item.get('distractor_quality', 0)}/1")
        print(f"  Rationale Match: {item.get('rationale_match', 0)}/1")
        print(f"  TOTAL: {item.get('total', 0)}/5")
    print("-" * 80)

print("\n✅ Section 11 complete!")


🔧 Rebuilding LangGraph with rubric_node...
✅ Graph rebuilt with rubric_node successfully!

Running pipeline with rubric evaluation...
✅ Objective: hypertension management in primary care
Searching PubMed for: hypertension AND management AND in AND primary AND care
Found 8 PMIDs
Successfully retrieved 8 PubMed documents
Searching Tavily for: hypertension AND management AND in AND primary AND care
Successfully retrieved 8 Tavily documents
🔎 PubMed=8 Tavily=8
Merged 8 PubMed + 8 Tavily = 16 total documents
Re-ranking 16 documents using Cohere
Successfully re-ranked 16 documents using Cohere
📈 Ranked=16
✅ Collection exists: pubmed_notes
✅ Collection exists: notes
Processing tavily document: 
Processing tavily document: 
Processing tavily document: hypertension-management.html
Processing tavily document: CIRCRESAHA.121.318083
Processing tavily document: 
Processing tavily document: changes-you-can-make-to-manage-high-blood-pressure
Processing tavily document: 2833283
Processing tavily docum

C:\Users\mspla\AppData\Local\Temp\ipykernel_17464\2003592004.py:35: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(


Retrieved 6 documents from Qdrant
No matching learning objective found in KG
Merged 6 documents, 2559 characters total
📚 Context=6
Generating 2 MCQ(s) using OpenAI GPT-4o-mini...
Successfully generated 2 MCQ(s)
🎯 MCQs=2
📊 Rubric: 2 MCQs evaluated, avg=4.5/5

RUBRIC EVALUATION RESULTS

📊 Overall: 2 MCQs, Avg Score: 4.5/5

📋 Per-Item Scores:
--------------------------------------------------------------------------------

MCQ 1: Which class of antihypertensive medication is preferred as f...
  Single Key: 1/1
  Clarity: 1/1
  Relevance: 0/1
  Distractor Quality: 1/1
  Rationale Match: 1/1
  TOTAL: 4/5

MCQ 2: What is the first step in the management of hypertension acc...
  Single Key: 1/1
  Clarity: 1/1
  Relevance: 1/1
  Distractor Quality: 1/1
  Rationale Match: 1/1
  TOTAL: 5/5
--------------------------------------------------------------------------------

✅ Section 11 complete!


## Section 12 — RAGAS Evaluation (Qdrant-Grounded)

**Goal.** Evaluate our pipeline using RAGAS on a golden test set of QA pairs synthesized strictly from Qdrant chunks (PubMed/Tavily).

**Key rules.**
- **No stubs**: require real Qdrant data and an OpenAI key for synthesis.
- **Grounded**: every QA must cite a specific Qdrant chunk (provenance).
- **Retrieval-first**: for evaluation, use the pipeline's retrieved contexts, not the gold chunk.
- **Skip, don't fake**: if retrieval returns no contexts, skip that item (do not fabricate contexts).

**Outputs.** Overall RAGAS metrics: faithfulness, answer relevancy, context precision, context recall. Counts and simple persona breakdown.


In [14]:
# --- Section 12: RAGAS Evaluation (Qdrant-Grounded, Clean Rewrite) ---

# 0) Imports / deps
import math, random, textwrap, json, re
from typing import List, Dict, Any
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    from datasets import Dataset
except Exception:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "ragas", "datasets", "evaluate"], check=True)
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    from datasets import Dataset

# 1) Configuration (tune as needed)
SYN_SAMPLES = 24                 # target number of QA items
RICH_POOL = max(12, SYN_SAMPLES//2)
PERSONAS = ["clinician", "educator", "student"]
MAX_CONTEXTS_FOR_EVAL = 3
MAX_CTX_CHARS = 800
EVAL_USE_KG = False              # keep False unless you guarantee a matching objective in the mini-KG

# 2) Ensure Qdrant client and collection name
def ensure_qdrant_client():
    """Return global Qdrant client created in Section 8; create if missing."""
    global client
    try:
        _ = client  # noqa
    except NameError:
        client = None
    if client is None:
        client = get_qdrant()   # uses your helper from Section 8
    return client

try:
    COLL_PUBMED
except NameError:
    COLL_PUBMED = "pubmed_notes"

def qdrant_count(collection: str) -> int:
    c = ensure_qdrant_client()
    try:
        res = c.count(collection, exact=True)
        return res.count if hasattr(res, "count") else int(res)
    except Exception:
        return 0

client = ensure_qdrant_client()
count = qdrant_count(COLL_PUBMED)
print(f"[RAGAS] Qdrant collection '{COLL_PUBMED}' count:", count)
if count < 10:
    raise RuntimeError("Not enough Qdrant points (<10). Run Section 10 once and Section 8 to store more chunks.")

# 3) Fetch Qdrant points (payload-driven)
def fetch_qdrant_points(limit=400) -> List[Dict[str, Any]]:
    c = ensure_qdrant_client()
    pts = c.scroll(collection_name=COLL_PUBMED, limit=limit, with_payload=True, with_vectors=False)
    items = pts[0] if isinstance(pts, tuple) else pts
    out = []
    for p in items:
        pay = p.payload or {}
        txt = pay.get("text", "")
        if not txt:
            continue
        out.append({
            "qid": getattr(p, "id", None),
            "source": pay.get("source", ""),
            "doc_id": pay.get("id", ""),
            "title": pay.get("title", ""),
            "url": pay.get("url", ""),
            "span_idx": pay.get("span_idx", -1),
            "text": txt,
        })
    return out

points = fetch_qdrant_points(limit=400)
if not points:
    raise RuntimeError("Qdrant returned no usable points with 'text' payload.")

# Prefer longer chunks for richer QA
points_sorted = sorted(points, key=lambda x: len(x["text"]), reverse=True)
rich = points_sorted[:min(RICH_POOL, len(points_sorted))]
print(f"[RAGAS] Selected {len(rich)} rich chunks.")

# 4) Grounded QA synthesis (LLM; no stubs)
def synthesize_qa_from_chunk(chunk: Dict[str, Any], persona: str) -> Dict[str, Any]:
    """
    Create one grounded QA strictly from the given chunk text.
    Requires OPENAI_API_KEY in api_keys; raises if missing. No stubs here.
    """
    if not api_keys.get("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY required for QA synthesis (no stubs).")
    from openai import OpenAI
    client_qa = OpenAI(api_key=api_keys["OPENAI_API_KEY"])
    passage = chunk["text"][:1800]
    prompt = f"""
Persona: {persona}
You are creating one grounded clinical/educational QA strictly from the provided passage.
Passage:
\"\"\"{passage}\"\"\"

Return JSON with:
- question: concise, unambiguous; answerable only from the passage
- answer: one-sentence factual answer derived from the passage (no external facts)
- evidence_quote: a short exact quote from the passage that supports the answer

JSON fields only: question, answer, evidence_quote
"""
    resp = client_qa.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=350
    )
    raw = resp.choices[0].message.content or ""
    m = re.search(r"\{.*\}", raw, re.S)
    data = json.loads(m.group(0)) if m else {"question": raw.strip(), "answer": "", "evidence_quote": ""}

    data.update({
        "persona": persona,
        "source": chunk["source"],
        "pmid_or_url": chunk["doc_id"] or chunk["url"],
        "chunk_title": chunk["title"],
        "chunk_span_idx": chunk["span_idx"],
    })
    return data

# 5) Build synthetic golden set (skip failed items; no stubs)
random.seed(13)
qa_items: List[Dict[str, Any]] = []
for _ in range(SYN_SAMPLES * 2):  # oversample attempts to compensate for skips
    if len(qa_items) >= SYN_SAMPLES:
        break
    base = random.choice(rich)
    persona = random.choice(PERSONAS)
    try:
        qa = synthesize_qa_from_chunk(base, persona)
        if qa.get("question") and qa.get("answer"):
            qa_items.append(qa)
    except Exception as e:
        print("[synthesis] skipped:", e)
        continue

print(f"[RAGAS] Synthesized {len(qa_items)} QA items.")
if len(qa_items) < max(8, SYN_SAMPLES//2):
    raise RuntimeError("Too few QA items synthesized. Increase Qdrant content or try again.")

# 6) Self-contained retrieval+answer for evaluation (Qdrant-only by default)
#    We make sure 'retrieve_qdrant' always has a client in scope and we avoid KG unless enabled.
try:
    _orig_retrieve_qdrant = retrieve_qdrant  # your original function
except NameError:
    _orig_retrieve_qdrant = None

def retrieve_qdrant_eval(query: str, k: int = 6):
    c = ensure_qdrant_client()
    if _orig_retrieve_qdrant is None:
        raise RuntimeError("Original retrieve_qdrant not found.")
    try:
        return _orig_retrieve_qdrant(query, k)
    except UnboundLocalError:
        globals()["client"] = c
        return _orig_retrieve_qdrant(query, k)

def eval_retrieve_and_answer(question: str, objective_topic: str) -> dict:
    # Qdrant retrieval
    try:
        q_hits = retrieve_qdrant_eval(question, k=6)
    except Exception as e:
        print("[eval] retrieve_qdrant error:", e)
        q_hits = []

    # Optional KG retrieval
    kg_hits = []
    if EVAL_USE_KG:
        try:
            kg_hits = retrieve_kg({"topic": objective_topic}, k=4)
        except Exception as e:
            print("[eval] retrieve_kg error:", e)

    # Merge context
    try:
        docs = merge_context(q_hits, kg_hits, max_chars=2000)
    except Exception as e:
        print("[eval] merge_context error:", e)
        docs = []
    docs = docs[:MAX_CONTEXTS_FOR_EVAL]

    if not docs:
        return {"contexts": [], "response": ""}

    ctx_texts = [textwrap.shorten(d.get("text",""), width=MAX_CTX_CHARS) for d in docs if d.get("text")]

    # Short grounded answer (no stubs)
    if not api_keys.get("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY required to answer (no stubs).")
    from openai import OpenAI
    client_ans = OpenAI(api_key=api_keys["OPENAI_API_KEY"])
    msgs = [
        {"role": "system", "content": "Answer strictly from the provided context. If insufficient, respond 'unsure'."},
        {"role": "user", "content": f"Question: {question}\n\nContext:\n" + "\n\n---\n".join(ctx_texts)}
    ]
    resp = client_ans.chat.completions.create(
        model="gpt-4o-mini",
        messages=msgs,
        temperature=0.1,
        max_tokens=120
    )
    reply = (resp.choices[0].message.content or "").strip()
    return {"contexts": ctx_texts, "response": reply}

# 7) Build evaluable rows (skip items with no retrieved contexts)
objective_topic = qa_items[0].get("chunk_title") or "medical education topic"
rows = []
skipped = 0
for qa in qa_items:
    pa = eval_retrieve_and_answer(qa["question"], objective_topic)
    if not pa["contexts"]:
        skipped += 1
        continue
    rows.append({
        "question": qa["question"],
        "ground_truth": qa["answer"],
        "contexts": pa["contexts"],
        "response": pa["response"],
        "persona": qa["persona"],
        "source": qa["source"],
        "pmid_or_url": qa["pmid_or_url"],
    })

print(f"[RAGAS] Prepared {len(rows)} evaluable items (skipped {skipped} with no contexts).")
if len(rows) < 8:
    raise RuntimeError("Too few evaluable items with retrieved contexts. Add data or re-run Section 10/8.")

# 8) Evaluate with RAGAS
ds = Dataset.from_list(rows)
ragas_result = evaluate(
    ds,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)

# ---- RAGAS metrics extraction & display (handles scores=list and wide/narrow DF)

def get_ragas_summary(result, debug=True):
    """
    Return a dict with {faithfulness, answer_relevancy, context_precision, context_recall}
    across different ragas versions:
      - result.scores may be a LIST of per-metric objects/dicts
      - result.to_pandas() may be "narrow" (metric/score) or "wide" (columns per metric)
    """
    wanted = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

    # 1) Try DataFrame (narrow or wide)
    try:
        df = result.to_pandas()
        if debug:
            print("[ragas] df columns:", list(df.columns))
            print(df.head(5))
        # Narrow shape: columns include 'metric' and 'score'
        if {"metric", "score"}.issubset(df.columns):
            cand = {row["metric"]: float(row["score"]) for _, row in df.iterrows() if row.get("metric") in wanted}
            if cand:
                return cand
        # Wide shape: metric names appear as columns directly
        wide = {}
        for m in wanted:
            if m in df.columns:
                try:
                    # take mean or the first non-null
                    val = df[m].astype(float).mean() if df[m].ndim == 1 else float(df[m].iloc[0])
                    wide[m] = float(val)
                except Exception:
                    try:
                        # fallback: first numeric value
                        wide[m] = float(df[m].dropna().iloc[0])
                    except Exception:
                        pass
        if wide:
            return wide
    except Exception as e:
        if debug:
            print("[ragas] to_pandas() failed:", e)

    # 2) scores as LIST of metric results
    try:
        scores = getattr(result, "scores", None)
        if debug:
            print("[ragas] scores type:", type(scores))
        out = {}
        if isinstance(scores, list) and scores:
            for item in scores:
                # item can be an object or a dict
                # metric name
                name = None
                if hasattr(item, "name"):
                    name = getattr(item, "name", None)
                elif hasattr(item, "metric"):
                    name = getattr(item, "metric", None)
                elif isinstance(item, dict):
                    name = item.get("name") or item.get("metric")
                # metric value
                val = None
                for attr in ("score", "value", "overall"):
                    if hasattr(item, attr):
                        try:
                            val = float(getattr(item, attr))
                            break
                        except Exception:
                            pass
                    if isinstance(item, dict) and attr in item:
                        try:
                            val = float(item[attr])
                            break
                        except Exception:
                            pass
                if name and (val is not None):
                    out[name] = val
        # filter to wanted keys only, if found
        out = {k: v for k, v in out.items() if k in wanted}
        if out:
            return out
    except Exception as e:
        if debug:
            print("[ragas] scores(list) parse failed:", e)

    # 3) Last resort: probe attributes directly
    out = {}
    for name in wanted:
        try:
            val = getattr(result, name)
            if val is not None:
                out[name] = float(val)
        except Exception:
            pass
    return out

# ---- Compute & print summary
summary = get_ragas_summary(ragas_result, debug=True)
print("\n[RAGAS] Overall metrics:", summary)

# ---- Table + persona counts
def _fmt(x):
    try:
        return f"{float(x):.3f}"
    except Exception:
        return "—"

print("\n=== RAGAS METRICS ===")
for m in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    print(f"{m:>18}: {_fmt(summary.get(m))}")

from collections import Counter
cnt = Counter(r["persona"] for r in rows)
print("\n=== Persona counts ===")
for k, v in cnt.items():
    print(f"{k}: {v}")

print("\n✅ Section 12 metrics summarized.")


[RAGAS] Qdrant collection 'pubmed_notes' count: 25
[RAGAS] Selected 12 rich chunks.
[RAGAS] Synthesized 24 QA items.


C:\Users\mspla\AppData\Local\Temp\ipykernel_17464\2003592004.py:35: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(


Retrieved 6 documents from Qdrant
Merged 5 documents, 1537 characters total
Retrieved 6 documents from Qdrant
Merged 3 documents, 1981 characters total
Retrieved 6 documents from Qdrant
Merged 4 documents, 1493 characters total
Retrieved 6 documents from Qdrant
Merged 3 documents, 1979 characters total
Retrieved 6 documents from Qdrant
Merged 4 documents, 1493 characters total
Retrieved 6 documents from Qdrant
Merged 3 documents, 1981 characters total
Retrieved 6 documents from Qdrant
Merged 4 documents, 1493 characters total
Retrieved 6 documents from Qdrant
Merged 3 documents, 1979 characters total
Retrieved 6 documents from Qdrant
Merged 3 documents, 1979 characters total
Retrieved 6 documents from Qdrant
Merged 5 documents, 1537 characters total
Retrieved 6 documents from Qdrant
Merged 4 documents, 1946 characters total
Retrieved 6 documents from Qdrant
Merged 3 documents, 1981 characters total
Retrieved 6 documents from Qdrant
Merged 2 documents, 1943 characters total
Retrieved 6 

Evaluating:   0%|          | 0/96 [00:00<?, ?it/s]

[ragas] df columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
                                          user_input  \
0  What is the initial treatment recommendation f...   
1  What organization developed worldwide practice...   
2  What was the improvement in BP control during ...   
3  What is Gary Edward Sander's role at Tulane Un...   
4  What was the improvement in BP control during ...   

                                  retrieved_contexts  \
0  [* Hypertension: Management of patients on com...   
1  [The International Society of Hypertension (IS...   
2  [Measure accurately, Act rapidly, and Partner ...   
3  [**Gary Edward Sander, MD, PhD, FACC, FAHA, FA...   
4  [Measure accurately, Act rapidly, and Partner ...   

                                            response  \
0  The initial treatment recommendation for patie...   
1  The International Society of Hypertension (ISH...   
2

# Section 13 — Advanced Retrieval (Hybrid: Qdrant Dense + SQLite FTS5 Sparse with RRF)

**Goal.** Add a robust hybrid retriever:

- **Dense**: Qdrant (OpenAI embeddings)
- **Sparse**: SQLite FTS5 (full-text search)
- **Fusion**: Reciprocal Rank Fusion (RRF), then optional Cohere re-rank

**Why.** Dense handles paraphrase/semantics; sparse catches exact medical terms and phrasing. Fusing both usually improves recall and faithfulness.

**Design.**
- `RETRIEVER_MODE`: "dense" or "hybrid_rrf" (set here)
- `chunks_fts.sqlite`: separate FTS5 index of {doc_id, title, text}
- **Safe FTS**: sanitize query → MATCH … on error → fallback to LIKE (sparse never fails hard)
- **Entrypoint**: `retrieve_for_pipeline(query, k)` → used by Sections 14 (orchestration) and 15 (evaluation)

**Note.** This section reuses earlier helpers (`retrieve_qdrant`, `rerank_cohere`, `ensure_qdrant_client`, `COLL_PUBMED`) if present. If a helper is missing, it degrades gracefully (e.g., sparse returns empty, hybrid still works via dense).



In [25]:
# --- Section 13: Advanced Retrieval (Hybrid Dense + Sparse with RRF) ---

import sqlite3, re
from typing import List, Dict, Any, Tuple

# 0) Toggle: choose retriever behavior for Sections 14/15
RETRIEVER_MODE = "hybrid_rrf"   # options: "dense" (Qdrant only) | "hybrid_rrf" (Qdrant + SQLite FTS5)

# 1) SQLite FTS5 index (kept separate from mini-KG to avoid coupling)
CHUNKS_FTS_DB = "chunks_fts.sqlite"

def _ensure_fts_db() -> sqlite3.Connection:
    conn = sqlite3.connect(CHUNKS_FTS_DB)
    cur = conn.cursor()
    cur.execute("""
    CREATE VIRTUAL TABLE IF NOT EXISTS chunks_fts USING fts5(
      doc_id UNINDEXED,
      title,
      text,
      content=''
    );
    """)
    conn.commit()
    return conn

def _fts_count(conn) -> int:
    try:
        cur = conn.cursor()
        cur.execute("SELECT count(*) FROM chunks_fts;")
        return int(cur.fetchone()[0])
    except Exception:
        return 0

def _fts_backfill_from_qdrant(conn, limit: int = 800):
    """
    One-time backfill of FTS index from Qdrant payloads.
    Requires: ensure_qdrant_client(), COLL_PUBMED.
    If not available, prints a warning and skips (hybrid still works via dense).
    """
    if 'ensure_qdrant_client' not in globals() or 'COLL_PUBMED' not in globals():
        print("[Section 13] Warning: ensure_qdrant_client/COLL_PUBMED missing; skipping FTS backfill.")
        return
    try:
        c = ensure_qdrant_client()  # reuse singleton created earlier
        rows = c.scroll(collection_name=COLL_PUBMED, limit=limit, with_payload=True, with_vectors=False)
        items = rows[0] if isinstance(rows, tuple) else rows
        to_insert = []
        for p in items:
            pay = getattr(p, "payload", None) or {}
            txt = pay.get("text", "")
            if not txt:
                continue
            to_insert.append((
                str(pay.get("id", "")),
                pay.get("title", "") or "(untitled)",
                txt
            ))
        if to_insert:
            cur = conn.cursor()
            cur.executemany("INSERT INTO chunks_fts (doc_id, title, text) VALUES (?, ?, ?)", to_insert)
            conn.commit()
            print(f"[Section 13] FTS backfill inserted {len(to_insert)} rows.")
        else:
            print("[Section 13] FTS backfill: no rows found in Qdrant.")
    except Exception as e:
        print("[Section 13] FTS backfill error:", e)

# 2) Sparse retrieval (FTS5) with safe MATCH + LIKE fallback
_FTS_STOP = {
    "the","and","or","of","to","a","in","for","on","with","by","as","at","is","are","was","were",
    "from","that","this","these","those","an","be","it","its","into","than","then","over","under"
}

def _fts_sanitize_query(q: str, max_terms: int = 12) -> str:
    """
    Build a safe FTS MATCH expression:
    - keep alphanumerics, lowercase
    - drop stopwords & 1-char tokens
    - AND-join quoted tokens => "hypertension" AND "primary" AND "care"
    """
    toks = re.findall(r"[A-Za-z0-9]+", (q or "").lower())
    toks = [t for t in toks if len(t) > 1 and t not in _FTS_STOP]
    toks = toks[:max_terms]
    return " AND ".join(f'"{t}"' for t in toks) if toks else '"health"'

def retrieve_sqlite_fts(query: str, k: int = 10) -> List[Dict[str, Any]]:
    conn = _ensure_fts_db()
    if _fts_count(conn) == 0:
        _fts_backfill_from_qdrant(conn, limit=800)

    cur = conn.cursor()
    safe = _fts_sanitize_query(query)

    try:
        cur.execute("""
            SELECT doc_id, title, snippet(chunks_fts, 2, '[', ']', '…', 10) as snip
            FROM chunks_fts
            WHERE chunks_fts MATCH ?
            ORDER BY rank
            LIMIT ?;
        """, (safe, k))
        rows = cur.fetchall()
    except Exception as e:
        # Fallback: LIKE scan to avoid hard failure on odd inputs
        print(f"[fts] MATCH failed ({e}); falling back to LIKE")
        like_seed = " ".join(re.findall(r"[A-Za-z0-9]+", query)[:4])
        like = f"%{like_seed}%"
        cur.execute("""
            SELECT doc_id, title, substr(text,1,400) as snip
            FROM chunks_fts
            WHERE text LIKE ?
            LIMIT ?;
        """, (like, k))
        rows = cur.fetchall()

    out = []
    for doc_id, title, snip in rows:
        # fetch full text
        try:
            cur2 = conn.cursor()
            cur2.execute("SELECT text FROM chunks_fts WHERE doc_id = ? LIMIT 1;", (doc_id,))
            r2 = cur2.fetchone()
            text = r2[0] if r2 else (snip or "")
        except Exception:
            text = snip or ""
        out.append({
            "source": "sqlite_fts",
            "id": str(doc_id),
            "title": title or "(no title)",
            "url": "",
            "text": text
        })
    return out

# 3) Dense retrieval wrapper (normalize to shared schema)
def retrieve_dense(query: str, k: int = 10) -> List[Dict[str, Any]]:
    """
    Uses your existing retrieve_qdrant(query, k) from earlier sections.
    Returns [{source,id,title,url,text}, ...].
    """
    if 'retrieve_qdrant' not in globals():
        print("[dense] retrieve_qdrant missing; returning empty list")
        return []
    try:
        hits = retrieve_qdrant(query, k=k)
    except Exception as e:
        print("[dense] retrieve_qdrant error:", e)
        hits = []
    norm = []
    for d in hits:
        if isinstance(d, dict):
            norm.append({
                "source": d.get("source", "qdrant"),
                "id": d.get("id", ""),
                "title": d.get("title", ""),
                "url": d.get("url", ""),
                "text": d.get("text", "")
            })
        else:
            # Doc object-like
            norm.append({
                "source": getattr(d, "source", "qdrant"),
                "id": getattr(d, "id", ""),
                "title": getattr(d, "title", ""),
                "url": getattr(d, "url", ""),
                "text": getattr(d, "text", "")
            })
    return norm

# 4) Reciprocal Rank Fusion (RRF)
def rrf_fuse(cands_a: List[Dict[str, Any]], cands_b: List[Dict[str, Any]], k: int = 10, K: int = 60) -> List[Dict[str, Any]]:
    """
    Fuse two ranked lists; higher score = better (1/(K+rank)).
    """
    def key(doc) -> Tuple[str, str]:
        return (doc.get("source",""), doc.get("id",""))
    ranks: Dict[Tuple[str, str], float] = {}
    for rank, d in enumerate(cands_a, start=1):
        ranks[key(d)] = ranks.get(key(d), 0.0) + 1.0 / (K + rank)
    for rank, d in enumerate(cands_b, start=1):
        ranks[key(d)] = ranks.get(key(d), 0.0) + 1.0 / (K + rank)
    unique: Dict[Tuple[str,str], Dict[str,Any]] = {}
    for lst in (cands_a, cands_b):
        for d in lst:
            unique.setdefault(key(d), d)
    fused = sorted(unique.values(), key=lambda d: ranks.get(key(d), 0.0), reverse=True)
    return fused[:k]

# 5) Hybrid retriever (dense + sparse + optional rerank)
def retrieve_hybrid(query: str, k: int = 10) -> List[Dict[str, Any]]:
    dense = retrieve_dense(query, k=max(k, 10))
    # Sparse must not raise; handle internally
    try:
        sparse = retrieve_sqlite_fts(query, k=max(k, 10))
    except Exception as e:
        print("[hybrid] sparse (FTS) failed; using dense only:", e)
        sparse = []

    fused = rrf_fuse(dense, sparse, k=max(2*k, 20))

    # Optional: Cohere rerank if available
    if 'rerank_cohere' in globals():
        try:
            reranked = rerank_cohere(query, fused, top_k=k)
            return reranked
        except Exception as e:
            print("[hybrid] rerank_cohere failed; returning fused:", e)
            return fused[:k]
    return fused[:k]

# 6) Unified entrypoint used by Sections 14 & 15
def retrieve_for_pipeline(query: str, k: int = 10) -> List[Dict[str, Any]]:
    if RETRIEVER_MODE == "hybrid_rrf":
        return retrieve_hybrid(query, k=k)
    return retrieve_dense(query, k=k)

print(f"✅ Section 13 ready. RETRIEVER_MODE = {RETRIEVER_MODE} | Use retrieve_for_pipeline(query, k)")


✅ Section 13 ready. RETRIEVER_MODE = hybrid_rrf | Use retrieve_for_pipeline(query, k)


# Section 14 — Orchestration with Swappable Retriever (LangGraph)

**Goal.** Run the end-to-end pipeline (clarify → dual search → merge+rerank → store → retrieve via retrieve_for_pipeline → MCQ → rubric) while allowing retriever swapping (Qdrant dense vs Hybrid RRF) defined in Section 13.

**How.** We reuse all prior utilities. The only change from earlier orchestration: the retrieve step now calls retrieve_for_pipeline(query, k); KG is optional.

**Outputs.**
- Console summary of counts (pub/web/ranked/context/mcqs).
- Printed MCQs (stem, options, answer, rationale).
- Rubric result if available.
- Shows which retriever mode ran (uses RETRIEVER_MODE from Section 13).


In [26]:
# --- Section 14: Orchestration (Swappable Retriever via retrieve_for_pipeline) ---

from typing import TypedDict, List, Dict, Any
try:
    from langgraph.graph import StateGraph, END
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "langgraph"], check=True)
    from langgraph.graph import StateGraph, END

# Use existing flags if present; otherwise defaults
try:
    RETRIEVER_MODE
except NameError:
    RETRIEVER_MODE = "dense"  # defined in Section 13 normally

# Optional KG usage in orchestration (toggle easily)
try:
    ORCH_USE_KG
except NameError:
    ORCH_USE_KG = False  # set True if your mini-KG is populated for this topic

# ---- Orchestrator state (reuse if defined; else define compatible) ----
try:
    OrchestratorState  # type: ignore
except NameError:
    class OrchestratorState(TypedDict, total=False):
        objective: Dict[str, Any]
        query: str
        pub: List[Dict[str, Any]]
        web: List[Dict[str, Any]]
        ranked: List[Dict[str, Any]]
        context: List[Dict[str, Any]]
        mcqs: List[Dict[str, Any]]
        rubric: Dict[str, Any]
        validation: Dict[str, Any]
        used_fallback: bool

# ---- Node functions (thin wrappers reusing your earlier utilities) ----
def clarify_node(state: OrchestratorState) -> OrchestratorState:
    obj = state.get("objective")
    if not obj:
        try:
            obj = clarify_objective()  # uses your existing clarifier (persona/topic/n_mcq)
        except NameError:
            print("[clarify] missing clarify_objective(); using default objective")
            obj = {"topic": "hypertension management in primary care", "n_mcq": 2}
    try:
        q = compose_query(obj["topic"])
    except NameError:
        print("[clarify] missing compose_query(); using topic as query")
        q = obj["topic"]
    print(f"🎯 Objective: {obj['topic']}")
    print(f"🔎 Query: {q}")
    return {**state, "objective": obj, "query": q}

def researchers_node(state: OrchestratorState) -> OrchestratorState:
    q = state["query"]
    try:
        pub = search_pubmed(q, n=8)
    except NameError:
        print("[research] missing search_pubmed(); using empty pub")
        pub = []
    try:
        web = search_tavily(q, n=8)
    except NameError:
        print("[research] missing search_tavily(); using empty web")
        web = []
    print(f"📚 PubMed={len(pub)} | 🌐 Tavily={len(web)}")
    return {**state, "pub": pub, "web": web}

def rank_node(state: OrchestratorState) -> OrchestratorState:
    q = state["query"]
    pub, web = state.get("pub", []), state.get("web", [])
    try:
        combined = merge_results(pub, web)
    except NameError:
        print("[rank] missing merge_results(); combining manually")
        combined = pub + web
    try:
        ranked = rerank_cohere(q, combined, top_k=8)
    except NameError:
        print("[rank] missing rerank_cohere(); passing combined as-is")
        ranked = combined[:8]
    except Exception as e:
        print("[rank] rerank error:", e)
        ranked = combined[:8]
    print(f"📈 Ranked={len(ranked)}")
    return {**state, "ranked": ranked[:8]}

def store_node(state: OrchestratorState) -> OrchestratorState:
    ranked = state.get("ranked", [])
    obj = state["objective"]
    if not ranked:
        return state
    # Convert to Doc if your chunk/store needs it
    try:
        Doc  # type: ignore
        DocWithScore  # type: ignore
    except NameError:
        # Define only if truly absent (avoid duplicate definitions)
        from typing import TypedDict
        class Doc(TypedDict, total=False):
            source: str; id: str; title: str; url: str; text: str
        class DocWithScore(Doc, total=False):
            score: float
    
    docs = []
    for d in ranked:
        docs.append(Doc(
            source=d["source"], id=d["id"], title=d["title"], url=d["url"], text=d["text"]
        ))
    # Vector store
    try:
        if 'client' in globals() and client is not None:
            chunk_and_upsert_qdrant(docs, client)
        else:
            chunk_and_upsert_qdrant(docs)
    except NameError:
        print("[store] missing chunk_and_upsert_qdrant(); skipping vector store")
    except Exception as e:
        print("[store] Qdrant upsert warn:", e)
    # Mini-KG
    try:
        kg_upsert_minimal(docs, obj)
    except NameError:
        print("[store] missing kg_upsert_minimal(); skipping KG")
    except Exception as e:
        print("[store] KG upsert warn:", e)
    print("💾 Stored to vector DB and mini-KG")
    return state

def retrieve_node(state: OrchestratorState) -> OrchestratorState:
    q = state["query"]
    # 🔁 Swappable retriever: defined in Section 13
    try:
        q_hits = retrieve_for_pipeline(q, k=6)
        print(f"🔎 Retriever[{RETRIEVER_MODE}] hits={len(q_hits)}")
    except NameError:
        print("[retrieve] missing retrieve_for_pipeline(); using empty hits")
        q_hits = []
    except Exception as e:
        print("[retrieve] pipeline retriever error:", e)
        q_hits = []
    kg_hits = []
    if ORCH_USE_KG:
        try:
            kg_hits = retrieve_kg(state["objective"], k=4)
            print(f"🗂️ KG hits={len(kg_hits)}")
        except NameError:
            print("[retrieve] missing retrieve_kg(); skipping KG")
        except Exception as e:
            print("[retrieve] KG warn:", e)
    try:
        context = merge_context(q_hits, kg_hits, max_chars=6000)
    except NameError:
        print("[retrieve] missing merge_context(); using q_hits as context")
        context = q_hits
    except Exception as e:
        print("[retrieve] merge warn:", e)
        context = []
    print(f"🧩 Context docs={len(context)}")
    return {**state, "context": context}

def context_router(state: OrchestratorState) -> str:
    ctx = state.get("context") or []
    return "mcq_node" if ctx else "fallback_node"

def fallback_node(state: OrchestratorState) -> OrchestratorState:
    topic = state["objective"]["topic"]
    try:
        paras = llm_fallback_context(topic)[:3]
    except NameError:
        print("[fallback] missing llm_fallback_context(); using topic as fallback")
        paras = [f"Fallback context for {topic}"]
    except Exception as e:
        print("[fallback] fallback error:", e)
        paras = [f"Fallback context for {topic}"]
    
    try:
        Doc  # type: ignore
    except NameError:
        from typing import TypedDict
        class Doc(TypedDict, total=False):
            source: str; id: str; title: str; url: str; text: str
    
    fb = [Doc(source="llm_fallback", id=f"fb-{i}", title=f"LLM Fallback {i+1}", url="", text=p) for i, p in enumerate(paras)]
    print(f"🛟 Fallback docs={len(fb)}")
    return {**state, "context": fb, "used_fallback": True}

def mcq_node(state: OrchestratorState) -> OrchestratorState:
    # Convert to DocWithScore (uniform score=1.0)
    try:
        DocWithScore  # type: ignore
    except NameError:
        from typing import TypedDict
        class DocWithScore(TypedDict, total=False):
            source: str; id: str; title: str; url: str; text: str; score: float
    
    ranked_like = [DocWithScore(**{**d, "score": 1.0}) for d in state["context"]]
    try:
        mcqs = generate_mcqs(state["objective"], ranked_like)
    except NameError:
        print("[mcq] missing generate_mcqs(); using empty MCQs")
        mcqs = []
    except Exception as e:
        print("[mcq] generation error:", e)
        mcqs = []
    
    if not mcqs:
        raise RuntimeError("MCQ generation returned no items.")
    print(f"❓ MCQs={len(mcqs)}")
    return {**state, "mcqs": mcqs}

def rubric_node(state: OrchestratorState) -> OrchestratorState:
    # Optional rubric agent if you defined it earlier
    try:
        report = evaluate_mcqs_with_rubric(state["mcqs"], state.get("context", []))
    except NameError:
        report = {"status": "skipped", "reason": "rubric agent not implemented"}
    except Exception as e:
        report = {"status": "error", "error": str(e)}
    print(f"📋 Rubric status={report.get('status','n/a')}")
    return {**state, "rubric": report}

# ---- Build & run graph ----
print("🏗️ Building Section 14 graph...")
g = StateGraph(OrchestratorState)
g.add_node("clarify", clarify_node)
g.add_node("research", researchers_node)
g.add_node("rank", rank_node)
g.add_node("store", store_node)
g.add_node("retrieve", retrieve_node)
g.add_node("fallback", fallback_node)
g.add_node("mcq", mcq_node)
g.add_node("rubric", rubric_node)

g.set_entry_point("clarify")
g.add_edge("clarify", "research")
g.add_edge("research", "rank")
g.add_edge("rank", "store")
g.add_edge("store", "retrieve")
g.add_conditional_edges("retrieve", context_router, {
    "mcq_node": "mcq",
    "fallback_node": "fallback"
})
g.add_edge("fallback", "mcq")
g.add_edge("mcq", "rubric")
g.add_edge("rubric", END)

app14 = g.compile()
print("✅ Section 14 graph ready.")

# ---- Invoke once with an explicit objective (or let clarify ask) ----
print("\n🚀 Running Section 14 orchestration...")
initial_state: OrchestratorState = {
    "objective": {"topic": "hypertension management in primary care", "n_mcq": 2}
}
final_state = app14.invoke(initial_state)

# ---- Summary & MCQs ----
print("\n📊 ORCHESTRATION SUMMARY")
summary = {
    "retriever_mode": RETRIEVER_MODE,
    "pub": len(final_state.get("pub", [])),
    "web": len(final_state.get("web", [])),
    "ranked": len(final_state.get("ranked", [])),
    "context": len(final_state.get("context", [])),
    "mcqs": len(final_state.get("mcqs", [])),
    "used_fallback": final_state.get("used_fallback", False),
    "rubric_status": final_state.get("rubric", {}).get("status", "n/a")
}
print(summary)

mcqs = final_state.get("mcqs", [])
if mcqs:
    print(f"\n🎯 GENERATED {len(mcqs)} MCQ(S) [mode={RETRIEVER_MODE}]")
    print("=" * 60)
    for i, q in enumerate(mcqs, 1):
        print(f"\nMCQ {i}: {q['stem']}")
        for idx, opt in enumerate(q["options"]):
            print(f"  {chr(65+idx)}. {opt}")
        correct = chr(65 + q['answer_idx'])
        print(f"  ✅ Correct: {correct}")
        print(f"  📝 Rationale: {q['rationale']}")
        print("-" * 40)
else:
    print("\n❌ No MCQs generated")


🏗️ Building Section 14 graph...
✅ Section 14 graph ready.

🚀 Running Section 14 orchestration...
🎯 Objective: hypertension management in primary care
🔎 Query: hypertension AND management AND in AND primary AND care
Searching PubMed for: hypertension AND management AND in AND primary AND care
Found 8 PMIDs
Successfully retrieved 8 PubMed documents
Searching Tavily for: hypertension AND management AND in AND primary AND care
Successfully retrieved 8 Tavily documents
📚 PubMed=8 | 🌐 Tavily=8
Merged 8 PubMed + 8 Tavily = 16 total documents
Re-ranking 16 documents using Cohere
Successfully re-ranked 16 documents using Cohere
📈 Ranked=16
✅ Collection exists: pubmed_notes
✅ Collection exists: notes
Processing tavily document: 
Processing tavily document: 
Processing tavily document: hypertension-management.html
Processing tavily document: 
Processing tavily document: changes-you-can-make-to-manage-high-blood-pressure
Processing tavily document: hypertension-in-adults-initial-drug-therapy
Proce

C:\Users\mspla\AppData\Local\Temp\ipykernel_17464\2003592004.py:35: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(


Retrieved 10 documents from Qdrant
Re-ranking 11 documents using Cohere
Successfully re-ranked 11 documents using Cohere
🔎 Retriever[hybrid_rrf] hits=11
Merged 11 documents, 4650 characters total
🧩 Context docs=11
Generating 2 MCQ(s) using OpenAI GPT-4o-mini...
Successfully generated 2 MCQ(s)
❓ MCQs=2
📋 Rubric status=skipped

📊 ORCHESTRATION SUMMARY
{'retriever_mode': 'hybrid_rrf', 'pub': 8, 'web': 8, 'ranked': 8, 'context': 11, 'mcqs': 2, 'used_fallback': False, 'rubric_status': 'skipped'}

🎯 GENERATED 2 MCQ(S) [mode=hybrid_rrf]

MCQ 1: What is the recommended initial drug therapy for patients with hypertension who are self-identified as Black?
  A. Angiotensin-converting enzyme (ACE) inhibitors
  B. Beta-blockers
  C. Calcium channel blockers
  D. Thiazide-like diuretics
  E. Diuretics
  ✅ Correct: C
  📝 Rationale: Calcium channel blockers or thiazide-like diuretics are preferred as initial therapy in self-identified Black patients due to their superior efficacy in lowering blood pre

# --- Section 15: RAGAS Evaluation (Retriever Comparison Mode) ---

This section evaluates retrieval and answer quality using **RAGAS** metrics for the current retriever mode (`dense`, `hybrid_rrf`, etc.).  
It uses the same golden test set generation logic as Section 12, but reuses the new unified retriever from **Section 13 (`retrieve_for_pipeline`)**.  

**Workflow:**
1. Pulls questions/answers from synthetic Qdrant-grounded set.  
2. Retrieves supporting context via `retrieve_for_pipeline()` based on current `RETRIEVER_MODE`.  
3. Answers strictly from the retrieved context.  
4. Computes RAGAS metrics — *faithfulness*, *answer relevancy*, *context precision*, *context recall*.  
5. Prints a compact comparison summary for each mode.

In [27]:
# --- Section 15: RAGAS Evaluation (Retriever Comparison Mode) ---

import textwrap, random, json
from typing import Dict, List, Any
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

print(f"\n=== Running RAGAS Evaluation for retriever mode: {RETRIEVER_MODE} ===")

# 1. Verify retriever exists
if 'retrieve_for_pipeline' not in globals():
    raise RuntimeError("❌ retrieve_for_pipeline not found. Run Section 13 first.")

# 2. Build evaluation samples (reuse QA items if still in memory)
try:
    qa_items
except NameError:
    raise RuntimeError("❌ qa_items not found. Run Section 12 or regenerate testset first.")

# 3. Evaluate retrieval and response generation
rows = []
skipped = 0

for qa in qa_items:
    q = qa["question"]
    gt = qa["answer"]

    try:
        # Retrieve context using the current retriever mode
        q_hits = retrieve_for_pipeline(q, k=6)
        ctx_texts = [textwrap.shorten(d.get("text", ""), width=800) for d in q_hits if d.get("text")]
    except Exception as e:
        print(f"[retrieve] error: {e}")
        ctx_texts = []

    if not ctx_texts:
        skipped += 1
        continue

    # Generate an answer grounded only in the retrieved context
    from openai import OpenAI
    if not api_keys.get("OPENAI_API_KEY"):
        raise RuntimeError("Missing OPENAI_API_KEY for answer synthesis.")

    llm_client = OpenAI(api_key=api_keys["OPENAI_API_KEY"])
    prompt = f"""Answer the question using only the provided context.
If context is insufficient, reply 'unsure'.
Question: {q}

Context:
{chr(10).join(ctx_texts)}"""

    resp = llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=120
    )
    reply = (resp.choices[0].message.content or "").strip()

    rows.append({
        "question": q,
        "ground_truth": gt,
        "contexts": ctx_texts,
        "response": reply,
        "persona": qa.get("persona", "unknown"),
        "retriever_mode": RETRIEVER_MODE
    })

print(f"[RAGAS] Prepared {len(rows)} evaluable items (skipped {skipped}).")

# 4. Evaluate with RAGAS
if len(rows) < 6:
    raise RuntimeError("Too few evaluable items. Add more Qdrant data or re-run Section 12.")

dataset = Dataset.from_list(rows)
ragas_result = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)

# 5. Extract metrics safely - FIXED VERSION
summary = {}
try:
    # Try the pandas approach first
    df = ragas_result.to_pandas()
    if not df.empty:
        summary = {row["metric"]: float(row["score"]) for _, row in df.iterrows()}
    else:
        raise ValueError("Empty dataframe")
except Exception as e1:
    print(f"[debug] pandas approach failed: {e1}")
    try:
        # Try the scores attribute
        scores = getattr(ragas_result, "scores", None)
        if scores is not None:
            if isinstance(scores, dict):
                summary = {k: float(v) for k, v in scores.items()}
            elif isinstance(scores, list):
                # Handle list of score dictionaries
                summary = {}
                for score_dict in scores:
                    if isinstance(score_dict, dict):
                        summary.update({k: float(v) for k, v in score_dict.items()})
            else:
                print(f"[debug] scores type: {type(scores)}, value: {scores}")
    except Exception as e2:
        print(f"[debug] scores approach failed: {e2}")
        # Last resort: try to extract from the result object directly
        try:
            result_dict = ragas_result.__dict__ if hasattr(ragas_result, '__dict__') else {}
            print(f"[debug] result object keys: {list(result_dict.keys())}")
        except Exception as e3:
            print(f"[debug] direct extraction failed: {e3}")

print("\n📊 RAGAS Metric Summary:")
for k in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    val = summary.get(k, None)
    print(f"{k:>18}: {val:.3f}" if val else f"{k:>18}: —")

# 6. Persona and retriever summary
from collections import Counter
cnt = Counter(r["persona"] for r in rows)
print("\n=== Persona counts ===")
for k, v in cnt.items():
    print(f"{k}: {v}")

print(f"\n✅ Evaluation complete for retriever mode: {RETRIEVER_MODE}")

# Optional: store for later comparison (dense vs hybrid)
if 'ragas_comparison' not in globals():
    ragas_comparison = {}
ragas_comparison[RETRIEVER_MODE] = summary
print("\n💾 Stored metrics for comparison:", ragas_comparison)


=== Running RAGAS Evaluation for retriever mode: hybrid_rrf ===


C:\Users\mspla\AppData\Local\Temp\ipykernel_17464\2003592004.py:35: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(


Retrieved 10 documents from Qdrant
Re-ranking 10 documents using Cohere
Successfully re-ranked 10 documents using Cohere
Retrieved 10 documents from Qdrant
Re-ranking 10 documents using Cohere
Successfully re-ranked 10 documents using Cohere
Retrieved 10 documents from Qdrant
Re-ranking 10 documents using Cohere
Successfully re-ranked 10 documents using Cohere
Retrieved 10 documents from Qdrant
Re-ranking 10 documents using Cohere
Successfully re-ranked 10 documents using Cohere
Retrieved 10 documents from Qdrant
Re-ranking 10 documents using Cohere
Successfully re-ranked 10 documents using Cohere
Retrieved 10 documents from Qdrant
Re-ranking 10 documents using Cohere
Successfully re-ranked 10 documents using Cohere
Retrieved 10 documents from Qdrant
Re-ranking 10 documents using Cohere
Successfully re-ranked 10 documents using Cohere
Retrieved 10 documents from Qdrant
Re-ranking 10 documents using Cohere
Successfully re-ranked 10 documents using Cohere
Retrieved 10 documents from Qdra

Evaluating:   0%|          | 0/96 [00:00<?, ?it/s]

[debug] pandas approach failed: 'metric'

📊 RAGAS Metric Summary:
      faithfulness: 0.500
  answer_relevancy: 1.000
 context_precision: 0.778
    context_recall: 1.000

=== Persona counts ===
educator: 8
student: 10
clinician: 6

✅ Evaluation complete for retriever mode: hybrid_rrf

💾 Stored metrics for comparison: {'dense': {'faithfulness': 1.0, 'answer_relevancy': 1.0000000000000002, 'context_precision': 0.699999999965, 'context_recall': 1.0}, 'hybrid_rrf': {'faithfulness': 0.5, 'answer_relevancy': 1.0000000000000002, 'context_precision': 0.7777777777518519, 'context_recall': 1.0}}


## Section 16 : Comparing RAGAS evluation between dense retrieval vs hybrid dense-sparse retrieval

### 📊 RAGAS Comparison Summary

**1. What is `ragas_comparison`?**  
`ragas_comparison` is a global dictionary that stores RAGAS metric summaries (faithfulness, answer relevancy, context precision, and context recall) for each retriever mode tested — e.g., `"dense"` or `"hybrid_rrf"`.  
It lets you compare multiple retrievers side-by-side within one notebook session.

---

**2. How to use it**  
Each time you run Section 15 with a chosen retriever mode, the code adds or updates an entry:  

ragas_comparison[RETRIEVER_MODE] = summary

Set retriever_mode at the top of section 13 code cell. Choose 'dense' or 'hybrid_rrf'

RETRIEVER_MODE = "dense"
Then rerun Sections 13–15, and repeat with:

RETRIEVER_MODE = "hybrid_rrf"
Both results will be stored automatically.

---

**3. How to display results**
Use either of these to view all stored results:

import json
print(json.dumps(ragas_comparison, indent=2))

or as a table:

import pandas as pd
pd.DataFrame(ragas_comparison).T

output:\
{
  "dense": {
    "faithfulness": 0.72,
    "answer_relevancy": 0.81,
    "context_precision": 0.65,
    "context_recall": 0.77
  },
  "hybrid_rrf": {
    "faithfulness": 0.75,
    "answer_relevancy": 0.84,
    "context_precision": 0.70,
    "context_recall": 0.80
  }
}

---

**4. How to reset**
To clear previous results and start fresh:
ragas_comparison = {}

In [28]:
pd.DataFrame(ragas_comparison).T

,faithfulness,answer_relevancy,context_precision,context_recall
dense,1.0,1.0,0.700000,1.0
hybrid_rrf,0.5,1.0,0.777778,1.0
